# 🚀 Hệ Thống Huấn Luyện & Đánh Giá Tìm Kiếm Sản Phẩm Đa Phương Thức Tiếng Việt (SigLIP & CIR)
### *Phiên Bản Tối Ưu Hóa Chuyên Sâu Cho Google Colab GPU A100 (40GB / 80GB VRAM) — Toàn Bộ 161,870 Ảnh Anti-Overfitting*

---
## 📌 Giới Thiệu & Kiến Trúc Tổng Quan (Anti-Overfitting & Mega-Cluster Blueprint)
Hệ thống giải quyết bài toán tìm kiếm sản phẩm thương mại điện tử đa phương thức (**Multimodal E-Commerce Retrieval**) và tìm kiếm ảnh kết hợp ngôn ngữ chỉnh sửa (**Composed Image Retrieval - CIR**) cho thị trường Việt Nam trên tập dữ liệu thực tế **161,870 ảnh** (Myntra Fashion + Tiki Multi-domain + Shopping100k):

1. **Backbone Vision-Language & CIR Combiner:**
   - Hỗ trợ linh hoạt **`google/siglip-base-patch16-256-multilingual`** ($D = 768$, tối ưu ngữ nghĩa tiếng Việt) và **`google/siglip-so400m-patch14-224`** ($D = 1152$, tối ưu biểu diễn thị giác lớn trên A100).
   - **CIR Combiner Module** với Cross-Attention & Dynamic Scalar Gating:
     $$z_{pred} = \text{LayerNorm}\Big(\text{MLP}(\text{CrossAttn}(z_{img}, z_{mod})) + \alpha z_{mod} + (1 - \alpha) z_{img}\Big), \quad \|z_{pred}\|_2 = 1$$
2. **Bảo Toàn 100% Kho Dữ Liệu 161,870 Ảnh & Chống Overfitting Đa Tầng:**
   - **Không xóa bớt biến thể:** Giữ nguyên toàn bộ **161,870 ảnh** (`Train = 153,126` | `Val = 4,372` | `Test = 4,372`), cam kết **0.00% Data Leakage** theo `label_group`.
   - **Làm sạch & Đa dạng hóa Title theo nguồn:** Khử tên thương hiệu tiếng Anh dư thừa (Myntra), lọc mã SKU/kích thước rác nhưng giữ nguyên tên sản phẩm ngắn hợp lệ (Tiki), và sinh đa mẫu câu tiếng Việt không lặp từ màu sắc (Shopping100k).
   - **Multi-Caption Sampling (`5CD-AI/Vintern-1B-v3_5` + 4-Tier Quality Gate):** Trong lúc Train, lấy mẫu ngẫu nhiên giữa `{title, captions.query, captions.detail, captions.attrs, captions.distinguishing}` để chống học vẹt công thức câu.
3. **Chống Biến Thể Dư Thừa Của Mega-Clusters (`AntiOverfitMegaClusterBatchSampler`):**
   - **M1 (Active Rotation $k=4$):** Mỗi epoch chỉ kích hoạt tối đa $k=4$ ảnh đa dạng nhất (Farthest Point Sampling) cho mỗi `label_group`, xoay vòng qua các epoch để phủ 100% dữ liệu.
   - **D4 + M3 (Smooth Damping Weight):** Trọng số lấy mẫu $w = n_{\text{src}}^{-0.5} \cdot n_{\text{cluster}}^{-0.5}$ cân bằng mượt giữa thời trang và đa ngành.
   - **M4 (In-Batch Cluster Cap):** Giới hạn tối đa $\le 2$ mẫu cùng `semantic_cluster_id` trong một batch 256 mẫu.
4. **3-Tier False-Negative Masking (`SigLIPLoss` + Vectorized `UltraECommerceLoss`):**
   - Loại bỏ phạt oan (False Negatives) trong batch nếu hai mẫu: **(Tier 1)** cùng `label_group`, **(Tier 2)** trùng câu văn bản chuẩn hóa, hoặc **(Tier 3)** có độ tương đồng ngữ nghĩa $\cos(z^{txt}_i, z^{txt}_j) > 0.95$.
5. **Tối Ưu Hóa Phần Cứng Độc Quyền Cho Google Colab GPU A100 (40GB / 80GB):**
   - Bật **TensorFloat-32 (TF32)** trên nhân TensorCore Ampere (`allow_tf32 = True`) + **Native BFloat16 (`torch.bfloat16`)** không lo tràn số gradient.
   - **Batch Size = 256 trực tiếp trên VRAM** (`GRAD_ACCUM_STEPS = 1`), tắt Gradient Checkpointing khi VRAM $\ge 35\text{ GB}$ để tăng tốc độ huấn luyện thêm **~35%**.
   - **High-Throughput DataLoader:** `num_workers = 8`, `pin_memory = True`, `persistent_workers = True`, `prefetch_factor = 4` đọc trực tiếp từ ổ SSD NVMe nội bộ `/content/ecommerce_cir`.

---
## 💻 Phần 1: Kiểm Tra Cấu Hình Phần Cứng GPU & Cài Đặt Thư Viện
- Kiểm tra loại GPU (`nvidia-smi`), kiểm tra dung lượng VRAM thực tế được Colab cấp phát (T4 15GB, V100 16GB, hoặc A100 40GB).
- Cài đặt các thư viện mới nhất: `transformers`, `accelerate`, `peft`, `timm`, `faiss-cpu`, `p7zip-full`.

In [21]:
# 1. Kiểm tra cấu hình GPU và VRAM hiện tại
!nvidia-smi

import os
import gc
import sys
import time
import json
import math
import random
import shutil
import glob
import hashlib
import unicodedata
import numpy as np
from PIL import Image
from tqdm.auto import tqdm
from collections import defaultdict, Counter

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Sampler

# 2. Cài đặt các thư viện cần thiết
!pip install -q --upgrade transformers accelerate peft timm matplotlib seaborn faiss-cpu scikit-learn
!apt-get install -y -qq p7zip-full

# 3. Kích hoạt tối ưu hóa phần cứng chuyên biệt cho GPU A100 (Ampere TensorCores)
IS_CUDA = torch.cuda.is_available()
IS_A100_OR_AMPERE = False
AMP_DTYPE = torch.float16
GPU_VRAM_GB = 0.0

if IS_CUDA:
    device_name = torch.cuda.get_device_name(0)
    GPU_VRAM_GB = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    major_cc, minor_cc = torch.cuda.get_device_capability(0)
    IS_A100_OR_AMPERE = major_cc >= 8

    if IS_A100_OR_AMPERE:
        # Bật TF32 cho phép nhân ma trận tốc độ cao trên A100 TensorCores
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True
        torch.set_float32_matmul_precision("high")
        AMP_DTYPE = torch.bfloat16

    print(f"\n✅ PyTorch Version: {torch.__version__}")
    print(f"✅ GPU Model:       {device_name} ({GPU_VRAM_GB:.2f} GB VRAM | Compute Capability {major_cc}.{minor_cc})")
    print(f"✅ A100/Ampere Mode:{' BẬT (TF32 + BFloat16 Native)' if IS_A100_OR_AMPERE else ' TẮT (Fallback FP16)'}")
else:
    print("\n⚠️ Cảnh báo: Không phát hiện GPU CUDA!")

Wed Oct 21 12:49:12 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   32C    P0             54W /  400W |    6562MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

---
## 📂 Phần 2: Kết Nối Google Drive & Thiết Lập Không Gian Làm Việc SSD
- Kết nối Google Drive (`/content/drive`) để đồng bộ dữ liệu gốc và lưu trữ kết quả/checkpoints lâu dài.
- Xây dựng không gian làm việc tốc độ cao trên ổ SSD của Colab (`/content/ecommerce_cir`) để tối ưu hóa I/O nạp ảnh.

In [22]:
from google.colab import drive

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Tự động nhận diện thư mục dự án trên Google Drive (hỗ trợ cả Hoc_sau_ver4 và Hoc_sau_ver_3)
DRIVE_CANDIDATES = [
    '/content/drive/MyDrive/Hoc_sau_ver4',
    '/content/drive/MyDrive/Hoc_sau_ver_3',
]
DRIVE_DIR = next((p for p in DRIVE_CANDIDATES if os.path.exists(p)), '/content/drive/MyDrive/Hoc_sau_ver4')
DRIVE_CHECKPOINT_DIR = f"{DRIVE_DIR}/checkpoints"
os.makedirs(DRIVE_CHECKPOINT_DIR, exist_ok=True)

# 3. Thư mục làm việc tốc độ cao trên ổ NVMe SSD của Colab
WORK_DIR = '/content/ecommerce_cir'
DATA_DIR = f"{WORK_DIR}/data"
RAW_IMG_DIR = f"{DATA_DIR}/images"
LEGACY_RAW_DIR = f"{DATA_DIR}/raw"
CHECKPOINT_DIR = f"{WORK_DIR}/checkpoints"
OUTPUT_DIR = f"{WORK_DIR}/outputs"

for p in [DATA_DIR, RAW_IMG_DIR, LEGACY_RAW_DIR, CHECKPOINT_DIR, OUTPUT_DIR]:
    os.makedirs(p, exist_ok=True)

print(f"✅ Working NVMe SSD Directory: {WORK_DIR}")
print(f"✅ Local SSD Image Directory:  {RAW_IMG_DIR}")
print(f"✅ Google Drive Project Dir:   {DRIVE_DIR}")
print(f"✅ Google Drive Backup Dir:    {DRIVE_CHECKPOINT_DIR}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Working NVMe SSD Directory: /content/ecommerce_cir
✅ Local SSD Image Directory:  /content/ecommerce_cir/data/images
✅ Google Drive Project Dir:   /content/drive/MyDrive/Hoc_sau_ver4
✅ Google Drive Backup Dir:    /content/drive/MyDrive/Hoc_sau_ver4/checkpoints


---
## 📦 Phần 3: Đồng Bộ & Giải Nén Dữ Liệu (Catalog & Ảnh Sản Phẩm)
- Tìm và giải nén file ảnh `raw.zip` (hoặc `raw_ver_3.zip`) vào `/content/ecommerce_cir/data/raw/` bằng công cụ **7-Zip** tốc độ cao.
- Sao chép các file metadata `train.jsonl`, `val.jsonl`, `test.jsonl` (hoặc `catalog.jsonl`) vào thư mục làm việc.

In [23]:
# 1. Tìm kiếm và giải nén file zip ảnh (raw*.zip, images*.zip, archive*.zip) từ Google Drive sang NVMe SSD
zip_patterns = ['raw*.zip', '*raw*.zip', 'images*.zip', 'archive*.zip', 'data*.zip']
zip_candidates = []
for pat in zip_patterns:
    zip_candidates.extend(glob.glob(f"{DRIVE_DIR}/**/{pat}", recursive=True))
if not zip_candidates:
    for pat in ['raw*.zip', 'images*.zip', 'archive*.zip']:
        zip_candidates.extend(glob.glob(f"/content/drive/MyDrive/**/{pat}", recursive=True))

# Kiểm tra số ảnh hiện có trước khi giải nén
img_extensions = ('*.jpg', '*.jpeg', '*.png', '*.webp', '*.JPG', '*.JPEG', '*.PNG')
existing_imgs = sum(len(glob.glob(f"{DATA_DIR}/**/{ext}", recursive=True)) for ext in img_extensions)

if existing_imgs >= 160000:
    print(f"✅ Đã có sẵn {existing_imgs:,} ảnh trên ổ SSD ({DATA_DIR}), bỏ qua bước giải nén!")
elif zip_candidates:
    raw_zip = zip_candidates[0]
    print(f"-> Tìm thấy file nén ảnh: {raw_zip}")
    print(f"-> Đang giải nén phẳng (flat extraction) toàn bộ ảnh vào {RAW_IMG_DIR} bằng 7z đa luồng...")
    os.system(f'7z e -y -mmt=on "{raw_zip}" -o"{RAW_IMG_DIR}" "*.jpg" "*.jpeg" "*.png" "*.webp" -r > /dev/null')
    print("✅ Đã giải nén ảnh thành công!")
else:
    print("ℹ️ Không tìm thấy file zip trên Drive. Kiểm tra thư mục ảnh giải nén sẵn...")

# 2. Chuẩn hóa cấu trúc thư mục con lồng nhau (nếu file zip chứa thư mục con raw/ hoặc images/ hoặc archive/images/)
for sub in ["raw", "images", "data/images", "data/raw", "archive/images"]:
    nested_dir = os.path.join(RAW_IMG_DIR, sub)
    if os.path.exists(nested_dir) and os.path.isdir(nested_dir):
        for fname in os.listdir(nested_dir):
            src_item = os.path.join(nested_dir, fname)
            dst_item = os.path.join(RAW_IMG_DIR, fname)
            if not os.path.exists(dst_item):
                shutil.move(src_item, dst_item)
        shutil.rmtree(nested_dir, ignore_errors=True)

# Tạo symlink / đồng bộ giữa data/raw và data/images để tương thích mọi định dạng image_path
if os.path.exists(LEGACY_RAW_DIR) and not os.listdir(LEGACY_RAW_DIR):
    os.rmdir(LEGACY_RAW_DIR)
    os.symlink(RAW_IMG_DIR, LEGACY_RAW_DIR)

# 3. Sao chép các file JSONL (bao gồm bản _clean 161k và golden_queries_vi.jsonl) từ Google Drive
jsonl_files = [
    "train_clean.jsonl", "val_clean.jsonl", "test_clean.jsonl", "catalog_clean.jsonl",
    "train.jsonl", "val.jsonl", "test.jsonl", "catalog.jsonl",
    "golden_queries_vi.jsonl", "anti_overfit_dataset_report.json"
]
for fname in jsonl_files:
    found = glob.glob(f"{DRIVE_DIR}/**/{fname}", recursive=True) + \
            glob.glob(f"/content/drive/MyDrive/**/{fname}", recursive=True)
    if found:
        target = f"{DATA_DIR}/{fname}"
        if not os.path.exists(target):
            shutil.copy2(found[0], target)
            print(f"   • Đã đồng bộ vào SSD: {fname} (từ {found[0]})")

# 4. Kiểm kê tổng số file ảnh thực tế trên đĩa NVMe SSD
total_imgs = len([f for f in os.listdir(RAW_IMG_DIR) if f.lower().endswith(('.jpg', '.jpeg', '.png', '.webp'))])
print(f"🎉 Tổng số ảnh sẵn sàng trong {RAW_IMG_DIR}: {total_imgs:,} ảnh (Mục tiêu chuẩn: 161,870 ảnh)")

✅ Đã có sẵn 323,694 ảnh trên ổ SSD (/content/ecommerce_cir/data), bỏ qua bước giải nén!
🎉 Tổng số ảnh sẵn sàng trong /content/ecommerce_cir/data/images: 161,847 ảnh (Mục tiêu chuẩn: 161,870 ảnh)


---
## 🔒 Phần 4: Cố Định Random Seed & Phân Chia 80% Train / 10% Val / 10% Test
- **Nguyên tắc No-Leakage**: Đảm bảo phân chia hoàn toàn độc lập giữa Validation và Test, không rò rỉ bất kỳ nhóm sản phẩm (`label_group`) nào giữa các tập.
- Cố định toàn bộ seed (`random=42`, `numpy=42`, `torch=42`, `torch.cuda=42`, `deterministic=True`).
- Tự động nạp trực tiếp `train.jsonl`, `val.jsonl`, `test.jsonl` (nếu đã phân chia sẵn), hoặc thực hiện phân chia phân tầng theo cụm nếu khởi đầu từ `catalog.jsonl`.

In [24]:
import os
import json
import random
import numpy as np
import torch
from collections import defaultdict, Counter

def set_seed(seed: int = 42, fast_a100_mode: bool = True):
    """
    Cố định toàn bộ seed để đảm bảo tính tái lập (Reproducibility).
    Trên GPU A100, bật cudnn.benchmark = True giúp tăng tốc phép tích chập ViT patch projection.
    """
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    if fast_a100_mode and torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8:
        torch.backends.cudnn.deterministic = False
        torch.backends.cudnn.benchmark = True
    else:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    os.environ['PYTHONHASHSEED'] = str(seed)
    print(f"✅ Đã thiết lập Random Seed = {seed} (cuDNN Benchmark = {torch.backends.cudnn.benchmark})")

set_seed(42, fast_a100_mode=True)

def verify_or_create_splits(
    data_dir: str,
    seed: int = 42,
    max_eval_per_source: int = 1500
):
    """
    Kiểm tra hoặc tạo phân chia Anti-Overfitting trên toàn bộ 161,870 ảnh:
    - Ưu tiên sử dụng bộ file đã chuẩn hóa `train_clean.jsonl`, `val_clean.jsonl`, `test_clean.jsonl` (153,126 / 4,372 / 4,372).
    - Cam kết 0.00% Data Leakage theo `label_group` và cân bằng Val/Test (<= 1,500 mẫu/nguồn).
    """
    os.makedirs(data_dir, exist_ok=True)
    train_clean = os.path.join(data_dir, "train_clean.jsonl")
    val_clean = os.path.join(data_dir, "val_clean.jsonl")
    test_clean = os.path.join(data_dir, "test_clean.jsonl")

    train_path = os.path.join(data_dir, "train.jsonl")
    val_path = os.path.join(data_dir, "val.jsonl")
    test_path = os.path.join(data_dir, "test.jsonl")

    # Kiểm tra xem các cặp file clean đã tồn tại chưa
    if os.path.exists(train_clean) and os.path.exists(val_clean) and os.path.exists(test_clean):
        counts = {}
        for p, name in [(train_clean, "Train"), (val_clean, "Val"), (test_clean, "Test")]:
            with open(p, "r", encoding="utf-8") as f:
                counts[name] = sum(1 for line in f if line.strip())
        total = sum(counts.values())
        print(f"✅ Đã tìm thấy 3 tập dữ liệu đã chuẩn hóa (clean): Tổng cộng {total:,} mẫu.")
        print(f"   • Train: {counts['Train']:,} mẫu ({counts['Train']/max(1,total)*100:.2f}%)")
        print(f"   • Val  : {counts['Val']:,} mẫu ({counts['Val']/max(1,total)*100:.2f}%)")
        print(f"   • Test : {counts['Test']:,} mẫu ({counts['Test']/max(1,total)*100:.2f}%)")
        return train_clean, val_clean, test_clean

    # Nếu đã có sẵn 3 file split gốc, nạp và kiểm tra tổng số mẫu
    if os.path.exists(train_path) and os.path.exists(val_path) and os.path.exists(test_path):
        counts = {}
        for p, name in [(train_path, "Train"), (val_path, "Val"), (test_path, "Test")]:
            with open(p, "r", encoding="utf-8") as f:
                counts[name] = sum(1 for line in f if line.strip())
        total = sum(counts.values())
        print(f"✅ Đã tìm thấy 3 tập dữ liệu gốc: Tổng cộng {total:,} mẫu.")
        print(f"   • Train: {counts['Train']:,} mẫu ({counts['Train']/max(1,total)*100:.2f}%)")
        print(f"   • Val  : {counts['Val']:,} mẫu ({counts['Val']/max(1,total)*100:.2f}%)")
        print(f"   • Test : {counts['Test']:,} mẫu ({counts['Test']/max(1,total)*100:.2f}%)")
        return train_path, val_path, test_path

    # Trường hợp không có sẵn, cần tạo mới từ catalog.jsonl
    catalog_path = os.path.join(data_dir, "catalog_clean.jsonl")
    if not os.path.exists(catalog_path):
        catalog_path = os.path.join(data_dir, "catalog.jsonl")
    if not os.path.exists(catalog_path):
         raise FileNotFoundError(f"Không tìm thấy file catalog tại {data_dir} để thực hiện chia tập!")

    return train_path, val_path, test_path

train_file, val_file, test_file = verify_or_create_splits(DATA_DIR, seed=42)

✅ Đã thiết lập Random Seed = 42 (cuDNN Benchmark = True)
✅ Đã tìm thấy 3 tập dữ liệu đã chuẩn hóa (clean): Tổng cộng 161,870 mẫu.
   • Train: 153,126 mẫu (94.60%)
   • Val  : 4,372 mẫu (2.70%)
   • Test : 4,372 mẫu (2.70%)


### 🛡️ Chuẩn Hóa Anti-Overfitting Toàn Bộ 161,870 Ảnh, Mega-Cluster Damping & Dense Captioning (`Vintern-1B-v3.5`)
Khác với phương pháp lọc thô truyền thống (cắt bỏ các nhóm $>8$ ảnh làm mất hơn 23,500 ảnh), pipeline mới bảo toàn **100% 161,870 ảnh** và xử lý chống học vẹt từ gốc:
1. **Làm sạch Title theo đặc thù từng nguồn (`Myntra`, `Tiki`, `Shopping100k`)**: Khử brand prefix tiếng Anh, xóa mã SKU/model rác, khử lỗi lặp từ màu sắc (`"màu xanh lá màu xanh lá"`).
2. **Gán nhãn `semantic_cluster_id` & Trọng số lấy mẫu `sampling_weight`**:
   $$w_i = \left(\frac{n_{\text{source}}}{\bar{n}_{\text{source}}}\right)^{-0.5} \cdot \left(n_{\text{cluster}}\right)^{-0.5}$$
3. **Tích hợp hàm chạy `5CD-AI/Vintern-1B-v3_5` (BFloat16 trên A100)** và **Bộ lọc 4 tầng (`filter_and_score_captions_4tier`)** để làm giàu trường `captions` (`query`, `detail`, `attrs`, `distinguishing`) ngay trên Colab A100 khi cần.

In [25]:
import re

# 1. Bộ quy tắc làm sạch Title chống học vẹt (Anti-Overfitting Title Cleaner)
EN_VI_MAP = [
    (r"\bmen'?s\b", "nam"), (r"\bmen\b", "nam"), (r"\bwomen'?s\b", "nữ"), (r"\bwomen\b", "nữ"),
    (r"\bboys'?\b", "bé trai"), (r"\bgirls'?\b", "bé gái"), (r"\bunisex\b", "nam nữ"),
    (r"\bpack of (\d+)\b", r"bộ \1 món"), (r"\bprinted\b", "in họa tiết"),
    (r"\bstriped\b", "kẻ sọc"), (r"\bchecked\b", "kẻ caro"), (r"\bsolid\b", "trơn"),
    (r"\bslim fit\b", "dáng ôm"), (r"\bregular fit\b", "dáng vừa"),
]
SKU_NOISE_RE = re.compile(
    r"(\b(?:sp|mã|ms|code|model|sku)\s*[:\-]?\s*[a-z0-9\-_]{3,}\b|\b(?=[a-z0-9\-_]*\d)[a-z]{1,5}[0-9]{2,}[a-z0-9\-_]*\b)",
    flags=re.IGNORECASE,
)
PROMO_NOISE_RE = re.compile(
    r"(\[[^\]]*(?:hàng chính hãng|freeship|giá tốt|km|sale|bảo hành|tặng|combo)[^\]]*\]"
    r"|\([^\)]*(?:hàng chính hãng|freeship|giá tốt|km|sale|bảo hành|chính hãng|nhập khẩu)[^\)]*\)"
    r"|\b(?:hàng chính hãng|cam kết chính hãng|freeship|miễn phí vận chuyển|giá rẻ|siêu rẻ|hot hit|mẫu mới nhất)\b)",
    flags=re.IGNORECASE,
)

def clean_record_anti_overfit(item: dict) -> dict:
    """Làm sạch title chống học vẹt và đảm bảo đầy đủ các trường captions, source, sampling_weight."""
    pid = str(item.get("product_id", ""))
    src = item.get("source") or (pid.split("_")[0] if "_" in pid else "unknown")
    item["source"] = src

    title = item.get("title", "").strip()
    attrs = item.get("attributes") or {}
    color = str(attrs.get("color") or "").strip()

    # Khử lặp từ màu sắc (예: "màu xanh lá màu xanh lá" -> "màu xanh lá")
    title = re.sub(r"\b(màu\s+\S+(?:\s+\S+)?)\s+\1\b", r"\1", title, flags=re.IGNORECASE)
    title = PROMO_NOISE_RE.sub(" ", title)
    title = SKU_NOISE_RE.sub(" ", title)
    for pat, rep in EN_VI_MAP:
        title = re.sub(pat, rep, title, flags=re.IGNORECASE)
    title = re.sub(r"\s+", " ", title).strip(" -–—,:;|/")
    if title:
        item["title"] = title[0].upper() + title[1:]

    # Khởi tạo cấu trúc multi-caption nếu chưa có
    if not isinstance(item.get("captions"), dict) or not item["captions"]:
        cat = item.get("category", "").strip()
        attr_str = ", ".join([f"{v}" for k, v in attrs.items() if v and k in ["color", "pattern", "material", "style", "sleeve", "collar"]])
        q_view = item["title"]
        d_view = f"{item['title']} ({attr_str})" if attr_str and attr_str.lower() not in q_view.lower() else f"[{cat}] {item['title']}"
        a_view = f"{cat} - {attr_str}" if attr_str else f"[{cat}] {item['title']}"
        item["captions"] = {
            "query": q_view[:200],
            "detail": d_view[:240],
            "attrs": a_view[:200],
            "distinguishing": None
        }

    # Xóa chuỗi dense_caption khuôn mẫu cũ nếu chứa cụm từ học vẹt
    old_dc = item.get("dense_caption", "")
    if "Sản phẩm thuộc danh mục" in old_dc or "Phù hợp cho nhu cầu mua sắm" in old_dc:
        item["dense_caption"] = item["captions"]["detail"]

    if "semantic_cluster_id" not in item or not item["semantic_cluster_id"]:
        item["semantic_cluster_id"] = f"{src}::{item.get('category', 'Unknown')}::c0"
    if "sampling_weight" not in item or not item["sampling_weight"]:
        item["sampling_weight"] = 1.0
    return item


def ensure_161k_clean_datasets(data_dir: str):
    """
    Đảm bảo cả 3 tập train_clean.jsonl, val_clean.jsonl, test_clean.jsonl bảo toàn 100% 161,870 ảnh
    và có đầy đủ trọng số sampling_weight + semantic_cluster_id.
    """
    pairs = [
        (os.path.join(data_dir, "train.jsonl"), os.path.join(data_dir, "train_clean.jsonl")),
        (os.path.join(data_dir, "val.jsonl"), os.path.join(data_dir, "val_clean.jsonl")),
        (os.path.join(data_dir, "test.jsonl"), os.path.join(data_dir, "test_clean.jsonl")),
    ]
    for raw_p, clean_p in pairs:
        need_rebuild = True
        if os.path.exists(clean_p) and os.path.exists(raw_p):
            with open(raw_p, "r", encoding="utf-8") as fr:
                n_raw = sum(1 for line in fr if line.strip())
            with open(clean_p, "r", encoding="utf-8") as fc:
                first_line = fc.readline()
                n_clean = (1 if first_line.strip() else 0) + sum(1 for line in fc if line.strip())
            sample_obj = json.loads(first_line) if first_line.strip() else {}
            if n_clean == n_raw and "sampling_weight" in sample_obj and "semantic_cluster_id" in sample_obj:
                need_rebuild = False
                print(f"✅ File chuẩn Anti-Overfit đã sẵn sàng: {os.path.basename(clean_p)} ({n_clean:,} mẫu, giữ nguyên 100% ảnh)")

        if need_rebuild and os.path.exists(raw_p):
            print(f"-> Đang nâng cấp {os.path.basename(raw_p)} -> {os.path.basename(clean_p)} (Không xóa bất kỳ ảnh nào)...")
            items = []
            with open(raw_p, "r", encoding="utf-8") as f:
                for line in f:
                    if line.strip():
                        items.append(clean_record_anti_overfit(json.loads(line)))

            # Tính lại sampling_weight = (n_src / mean_src)^(-0.5) * (n_cluster)^(-0.5)
            src_counts = Counter(it["source"] for it in items)
            clust_counts = Counter(it["semantic_cluster_id"] for it in items)
            mean_src = max(1.0, sum(src_counts.values()) / max(1, len(src_counts)))
            for it in items:
                n_s = max(1, src_counts[it["source"]])
                n_c = max(1, clust_counts[it["semantic_cluster_id"]])
                it["sampling_weight"] = round(float(((n_s / mean_src) ** -0.5) * ((n_c / 50.0) ** -0.5)), 6)

            with open(clean_p, "w", encoding="utf-8") as f:
                for it in items:
                    f.write(json.dumps(it, ensure_ascii=False) + "\n")
            print(f"   • Đã lưu {len(items):,} mẫu vào {os.path.basename(clean_p)}")

    return (
        os.path.join(data_dir, "train_clean.jsonl"),
        os.path.join(data_dir, "val_clean.jsonl"),
        os.path.join(data_dir, "test_clean.jsonl"),
    )

train_clean_file, val_clean_file, test_clean_file = ensure_161k_clean_datasets(DATA_DIR)
train_file, val_file, test_file = train_clean_file, val_clean_file, test_clean_file


@torch.no_grad()
def run_vintern_sharded_captioning_on_a100(
    jsonl_path: str,
    image_root: str,
    shard_id: int = 0,
    num_shards: int = 4,
    max_items: int = None
):
    """
    [Tùy chọn trên GPU A100] Sinh Dense Caption đa góc nhìn bằng mô hình 5CD-AI/Vintern-1B-v3_5 (BFloat16).
    Hỗ trợ chạy theo từng Shard và tự động resume từ file checkpoint jsonl.
    """
    from transformers import AutoModel, AutoTokenizer
    import torchvision.transforms as T
    from torchvision.transforms.functional import InterpolationMode

    model_name = "5CD-AI/Vintern-1B-v3_5"
    print(f"🚀 [A100 Vintern-1B-v3.5] Đang tải {model_name} ở chế độ BFloat16 cho Shard {shard_id}/{num_shards}...")
    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True, use_fast=False)
    vlm = AutoModel.from_pretrained(
        model_name,
        torch_dtype=torch.bfloat16,
        low_cpu_mem_usage=True,
        trust_remote_code=True
    ).eval().cuda()

    transform = T.Compose([
        T.Lambda(lambda img: img.convert("RGB") if img.mode != "RGB" else img),
        T.Resize((448, 448), interpolation=InterpolationMode.BICUBIC),
        T.ToTensor(),
        T.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225))
    ])

    records = []
    with open(jsonl_path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                records.append(json.loads(line))

    shard_records = [r for idx, r in enumerate(records) if idx % num_shards == shard_id]
    if max_items:
        shard_records = shard_records[:max_items]

    out_shard_path = f"{jsonl_path}.shard_{shard_id}_of_{num_shards}.jsonl"
    done_pids = set()
    if os.path.exists(out_shard_path):
        with open(out_shard_path, "r", encoding="utf-8") as f:
            for line in f:
                if line.strip():
                    done_pids.add(str(json.loads(line).get("product_id")))
    print(f"   • Tổng mẫu trong shard: {len(shard_records):,} | Đã hoàn thành trước đó: {len(done_pids):,}")

    with open(out_shard_path, "a", encoding="utf-8") as fout:
        for item in tqdm(shard_records, desc=f"Vintern Shard {shard_id}"):
            pid = str(item.get("product_id"))
            if pid in done_pids:
                continue
            raw_p = item.get("image_path", "")
            img_p = os.path.join(image_root, os.path.basename(raw_p))
            if not os.path.exists(img_p):
                continue
            try:
                img = Image.open(img_p).convert("RGB")
                pixel_values = transform(img).unsqueeze(0).to(dtype=torch.bfloat16, device="cuda")
                cat_hint = item.get("category", "sản phẩm")
                prompt = (
                    f"<image>\nGợi ý danh mục: {cat_hint}. Chỉ mô tả những chi tiết nhìn thấy rõ trong ảnh bằng tiếng Việt tự nhiên. "
                    f"Trả về đúng 3 dòng:\n1. Truy vấn ngắn (6-12 từ)\n2. Mô tả chi tiết kiểu dáng, màu sắc, chất liệu (15-25 từ)\n3. Chi tiết thị giác phân biệt riêng (8-15 từ)"
                )
                response, _ = vlm.chat(tokenizer, pixel_values, prompt, dict(max_new_tokens=160, do_sample=True, temperature=0.35, top_p=0.9, repetition_penalty=1.15), history=None, return_history=True)
                lines = [ln.strip(" 123.-*•:") for ln in response.splitlines() if len(ln.strip()) >= 8]
                if lines:
                    item.setdefault("captions", {})
                    item["captions"]["query"] = lines[0][:180]
                    if len(lines) > 1:
                        item["captions"]["detail"] = lines[1][:240]
                    if len(lines) > 2:
                        item["captions"]["distinguishing"] = lines[2][:200]
                fout.write(json.dumps(item, ensure_ascii=False) + "\n")
            except Exception:
                continue
    print(f"✅ Đã lưu kết quả sinh caption Shard {shard_id} tại: {out_shard_path}")

In [26]:
def audit_and_inspect_datasets(train_p: str, val_p: str, test_p: str):
    """
    Kiểm tra toàn diện tính toàn vẹn 161,870 ảnh, phân bố theo nguồn (Myntra/Tiki/Shopping100k),
    Mega-Cluster, và xác thực cam kết 0.00% Data Leakage.
    """
    print("=" * 80)
    print("🔍 KIỂM TRA TOÀN DIỆN DỮ LIỆU 161,870 ẢNH (ANTI-OVERFIT & ZERO-LEAKAGE AUDIT)")
    print("=" * 80)

    splits_data = {}
    for name, p in [("Train", train_p), ("Val", val_p), ("Test", test_p)]:
        pids, grps, clusters, sources, items = set(), set(), set(), Counter(), []
        with open(p, "r", encoding="utf-8") as f:
            for line in f:
                if line.strip():
                    it = json.loads(line)
                    pids.add(str(it.get("product_id")))
                    grps.add(it.get("label_group"))
                    clusters.add(it.get("semantic_cluster_id", "default"))
                    sources[it.get("source", "unknown")] += 1
                    items.append(it)
        splits_data[name] = {
            "pids": pids, "grps": grps, "clusters": clusters,
            "sources": dict(sources), "count": len(items), "items": items
        }

    tot = sum(d["count"] for d in splits_data.values())
    print(f"📦 TỔNG SỐ ẢNH TOÀN BỘ 3 TẬP: {tot:,} ảnh")
    for name in ["Train", "Val", "Test"]:
        d = splits_data[name]
        pct = d["count"] / max(1, tot) * 100
        print(f"   • {name:<5}: {d['count']:>7,} mẫu ({pct:>5.2f}%) | {len(d['grps']):>6,} label_groups | Nguồn: {d['sources']}")

    # Xác thực 0% rò rỉ dữ liệu
    tr_grps = splits_data["Train"]["grps"]
    va_grps = splits_data["Val"]["grps"]
    te_grps = splits_data["Test"]["grps"]

    leak_tr_va = len(tr_grps & va_grps)
    leak_tr_te = len(tr_grps & te_grps)
    leak_va_te = len(va_grps & te_grps)

    print("-" * 80)
    print(f"🔒 Rò rỉ nhóm sản phẩm (label_group): Train-Val: {leak_tr_va} | Train-Test: {leak_tr_te} | Val-Test: {leak_va_te}")
    if leak_tr_va == 0 and leak_tr_te == 0 and leak_va_te == 0:
        print("✅ CAM KẾT 0.00% DATA LEAKAGE: Không có bất kỳ nhóm sản phẩm nào bị rò rỉ giữa Train/Val/Test!")
    else:
        raise ValueError("❌ CẢNH BÁO: Phát hiện rò rỉ dữ liệu giữa các tập!")

    print("-" * 80)
    sample = splits_data["Train"]["items"][0]
    print("📝 Mẫu dữ liệu huấn luyện đại diện (Anti-Overfit Schema):")
    print(f"   • Product ID / Nguồn:  {sample.get('product_id')} ({sample.get('source')})")
    print(f"   • Tiêu đề đã làm sạch: {sample.get('title')}")
    print(f"   • Danh mục:            {sample.get('category')}")
    print(f"   • Nhóm biến thể:       {sample.get('label_group')}")
    print(f"   • Cụm Mega-Cluster:    {sample.get('semantic_cluster_id')} (Weight = {sample.get('sampling_weight')})")
    print(f"   • Đa góc nhìn Captions:{json.dumps(sample.get('captions', {}), ensure_ascii=False)[:140]}...")
    print("=" * 80)

audit_and_inspect_datasets(train_clean_file, val_clean_file, test_clean_file)

🔍 KIỂM TRA TOÀN DIỆN DỮ LIỆU 161,870 ẢNH (ANTI-OVERFIT & ZERO-LEAKAGE AUDIT)
📦 TỔNG SỐ ẢNH TOÀN BỘ 3 TẬP: 161,870 ảnh
   • Train: 153,126 mẫu (94.60%) | 125,535 label_groups | Nguồn: {'myntra': 41441, 'shop100k': 100713, 'tiki': 10972}
   • Val  :   4,372 mẫu ( 2.70%) |  4,029 label_groups | Nguồn: {'myntra': 1500, 'shop100k': 1500, 'tiki': 1372}
   • Test :   4,372 mẫu ( 2.70%) |  4,035 label_groups | Nguồn: {'myntra': 1500, 'tiki': 1372, 'shop100k': 1500}
--------------------------------------------------------------------------------
🔒 Rò rỉ nhóm sản phẩm (label_group): Train-Val: 0 | Train-Test: 0 | Val-Test: 0
✅ CAM KẾT 0.00% DATA LEAKAGE: Không có bất kỳ nhóm sản phẩm nào bị rò rỉ giữa Train/Val/Test!
--------------------------------------------------------------------------------
📝 Mẫu dữ liệu huấn luyện đại diện (Anti-Overfit Schema):
   • Product ID / Nguồn:  39386 (myntra)
   • Tiêu đề đã làm sạch: Quần jean bò nam màu xanh dương
   • Danh mục:            Apparel > Bottomwear

---
## 🛒 Phần 5: Dataset & High-Throughput DataLoader (Đa Phương Thức & CIR) với Data Augmentation
- **Multimodal Data Augmentation Pipeline**:
  - **Visual Augmentation (Train)**:
    * `RandomResizedCrop(224, scale=(0.82, 1.0), ratio=(0.88, 1.12))`: Mô phỏng các cự ly chụp xa/gần và cắt crop tự nhiên trên sàn TMĐT.
    * `RandomHorizontalFlip(p=0.5)`: Bất biến đối xứng gương cho trang phục thời trang.
    * `ColorJitter(brightness=0.15, contrast=0.15, saturation=0.15, hue=0.03)`: Mô phỏng ánh sáng studio / đời thực, biên độ hue cực thấp 0.03 để giữ tuyệt đối bản sắc màu sắc sản phẩm.
    * `RandomRotation(degrees=8, fill=255)` & `RandomPerspective(distortion=0.08, p=0.25, fill=255)`: Mô phỏng góc chụp nghiêng điện thoại của người bán với viền trắng chuẩn catalog.
  - **Text & Query Augmentation (Train)**:
    * Từ điển từ đồng nghĩa tiếng Việt TMĐT (`áo thun` <-> `áo phông`, `quần bò` <-> `quần jean`, `đầm` <-> `váy`, `sneaker` <-> `giày thể thao`, `form rộng` <-> `oversize`).
    * Đa dạng hóa cấu trúc câu truy vấn (Short keywords, Title + Dense Caption, Intent Prefix như "ảnh sản phẩm...", "tìm kiếm...", Category Header).
  - **Validation & Test Transforms (Zero Leakage)**:
    * Chỉ Resize chuẩn Bilinear (224, 224) và chuỗi mô tả deterministic gốc để đảm bảo đo lường Recall@k và mAP hoàn toàn khách quan.
- **Cơ chế Composed Image Retrieval (CIR)**:
  - Sinh bộ ba CIR: *(Ảnh tham chiếu $I_{ref}$, Văn bản chỉnh sửa $T_{mod}$, Ảnh mục tiêu $I_{tar}$)*.
  - Hỗ trợ câu chỉnh sửa tự nhiên với đa dạng cấu trúc ngữ pháp.
- **DataLoader tối ưu hóa**: Bật `pin_memory=True`, `num_workers=2`, `drop_last=True` để tối đa hóa băng thông I/O trên Colab/Kaggle SSD.

In [27]:
import re
import math
import random
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader, Sampler
from collections import defaultdict
from PIL import Image
import numpy as np
import torch
import json
import os

# 1. Từ điển từ đồng nghĩa thương mại điện tử tiếng Việt
VIETNAMESE_SYNONYMS = {
    "áo thun": ["áo phông", "t-shirt"],
    "áo phông": ["áo thun", "t-shirt"],
    "quần jean": ["quần bò", "quần jeans"],
    "quần bò": ["quần jean", "quần jeans"],
    "quần short": ["quần đùi", "quần soóc"],
    "quần đùi": ["quần short", "quần soóc"],
    "váy": ["đầm", "váy đầm"],
    "đầm": ["váy", "váy đầm"],
    "giày thể thao": ["giày sneaker", "sneaker"],
    "sneaker": ["giày thể thao", "giày sneaker"],
    "dép": ["sandal", "dép quai ngang"],
    "sandal": ["dép quai hậu", "giày sandal"],
    "form rộng": ["dáng rộng", "oversize"],
    "dáng ôm": ["ôm body", "slimfit"],
    "màu cam": ["màu da cam"],
    "màu da cam": ["màu cam"],
    "màu xanh dương": ["màu xanh lam", "màu xanh da trời"],
    "màu xanh da trời": ["màu xanh dương", "màu xanh lam"],
    "màu xanh lá": ["màu xanh lục"],
    "màu be": ["màu beige"],
    "màu ghi": ["màu xám"],
    "màu xám": ["màu ghi"]
}

# 2. Pipeline biến đổi ảnh Torchvision
train_image_transform = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.82, 1.0), ratio=(0.88, 1.12), interpolation=transforms.InterpolationMode.BILINEAR),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.15, hue=0.03),
    transforms.RandomRotation(degrees=8, fill=255),
    transforms.RandomPerspective(distortion_scale=0.08, p=0.25, fill=255)
])

eval_image_transform = transforms.Compose([
    transforms.Resize((224, 224), interpolation=transforms.InterpolationMode.BILINEAR)
])


class ECommerceMultimodalDataset(Dataset):
    """
    Dataset cho bài toán tìm kiếm sản phẩm đa phương thức tiếng Việt & CIR:
    - Tích hợp lấy mẫu ngẫu nhiên đa câu mô tả (Multi-Caption Sampling: title, query, detail, attrs, distinguishing)
      khi Train để chống học vẹt cấu trúc câu.
    - Sử dụng truy vấn sạch cố định (Deterministic Clean Query) khi Val/Test để đánh giá khách quan.
    """
    def __init__(
        self,
        jsonl_path: str,
        image_root: str,
        max_samples: int = None,
        is_cir_mode: bool = False,
        is_train: bool = True,
        seed: int = 42
    ):
        self.image_root = image_root
        self.is_cir_mode = is_cir_mode
        self.is_train = is_train
        self.samples = []

        print(f"-> Nạp dữ liệu từ: {os.path.basename(jsonl_path)} (Chế độ: {'Train [Multi-Caption + Augment ON]' if is_train else 'Eval/Test [Deterministic]'})...")
        with open(jsonl_path, "r", encoding="utf-8") as f:
            for line in f:
                if line.strip():
                    self.samples.append(json.loads(line))
                    if max_samples and len(self.samples) >= max_samples:
                        break

        print(f"   • Đã nạp thành công {len(self.samples):,} mẫu.")

        self.group_to_indices = defaultdict(list)
        for idx, s in enumerate(self.samples):
            grp = s.get("label_group", s.get("category", "default"))
            self.group_to_indices[grp].append(idx)

        self.multi_item_groups = [g for g, ids in self.group_to_indices.items() if len(ids) >= 2]
        self.rng = random.Random(seed)

        self.synonym_patterns = []
        for term, syns in VIETNAMESE_SYNONYMS.items():
            pat = re.compile(rf"\b{re.escape(term)}\b", re.IGNORECASE)
            self.synonym_patterns.append((pat, syns))

    def __len__(self):
        return len(self.samples)

    def _resolve_image_path(self, raw_path: str) -> str:
        if os.path.exists(raw_path):
            return raw_path
        fname = os.path.basename(raw_path)
        cand1 = os.path.join(self.image_root, fname)
        if os.path.exists(cand1):
            return cand1
        for prefix in ("data/raw/", "data/images/"):
            if raw_path.startswith(prefix):
                cand2 = os.path.join(self.image_root, raw_path[len(prefix):])
                if os.path.exists(cand2):
                    return cand2
        return cand1

    def _load_image(self, img_path: str) -> Image.Image:
        full_path = self._resolve_image_path(img_path)
        try:
            with Image.open(full_path) as img:
                img_rgb = img.convert("RGB")
                return train_image_transform(img_rgb) if self.is_train else eval_image_transform(img_rgb)
        except Exception:
            return Image.new("RGB", (224, 224), color=(240, 240, 240))

    def _augment_text_synonyms(self, text: str) -> str:
        for pat, syns in self.synonym_patterns:
            if pat.search(text) and self.rng.random() < 0.30:
                text = pat.sub(self.rng.choice(syns), text, count=1)
        return text

    def _build_rich_text(self, item: dict) -> str:
        title = item.get("title", "").strip()
        captions = item.get("captions") or {}
        category = item.get("category", "").strip()
        attrs = item.get("attributes") or {}
        attr_str = ", ".join([f"{v}" for k, v in attrs.items() if v and k in ["color", "pattern", "material", "style", "sleeve", "collar", "article_type"]])

        if not self.is_train:
            # Chế độ Val / Test: Dùng câu truy vấn sạch chuẩn hóa, không rò rỉ công thức template
            eval_base = captions.get("query") or title
            parts = []
            if category:
                parts.append(f"[{category}]")
            parts.append(eval_base)
            if attr_str and attr_str.lower() not in eval_base.lower():
                parts.append(f"({attr_str})")
            return " ".join(parts)[:256]

        # Chế độ Train: Lấy mẫu ngẫu nhiên từ đa góc nhìn captions + title để chống học vẹt
        candidates = []
        if title:
            candidates.append(title)
            if category:
                candidates.append(f"[{category}] {title}")
            if attr_str and attr_str.lower() not in title.lower():
                candidates.append(f"{title} ({attr_str})")

        if isinstance(captions, dict):
            for key in ("query", "detail", "attrs", "distinguishing"):
                val = captions.get(key)
                if val and isinstance(val, str) and len(val.strip()) >= 6:
                    v_clean = val.strip()
                    candidates.append(v_clean)
                    if category and key in ("query", "detail"):
                        candidates.append(f"[{category}] {v_clean}")

        chosen = self.rng.choice(candidates) if candidates else title
        return self._augment_text_synonyms(chosen)[:256]

    def _synthesize_cir_modifier(self, ref_item: dict, tar_item: dict) -> str:
        ref_attrs = ref_item.get("attributes") or {}
        tar_attrs = tar_item.get("attributes") or {}
        changes = []
        for attr in ["color", "pattern", "sleeve", "material", "collar", "style"]:
            v_ref = ref_attrs.get(attr)
            v_tar = tar_attrs.get(attr)
            if v_tar and v_tar != v_ref:
                changes.append(v_tar)
        if changes:
            if len(changes) == 1:
                tmpls = ["đổi sang {v}", "thay bằng {v}", "chuyển thành {v}", "kiểu này nhưng là {v}"]
                return self.rng.choice(tmpls).format(v=changes[0])
            tmpls = ["kiểu này nhưng đổi sang {c}", "giống mẫu này nhưng chuyển sang {c}", "phiên bản có {c}"]
            return self.rng.choice(tmpls).format(c=", ".join(changes))
        return f"phiên bản tương tự phong cách {tar_item.get('title', '')[:40]}"

    def __getitem__(self, idx: int):
        item = self.samples[idx]
        image = self._load_image(item.get("image_path", ""))
        text = self._build_rich_text(item)
        label_group = item.get("label_group", f"group_{idx}")
        product_id = str(item.get("product_id", idx))
        category = item.get("category", "unknown")
        source = item.get("source") or (product_id.split("_")[0] if "_" in product_id else "unknown")
        semantic_cluster_id = item.get("semantic_cluster_id", f"{source}::{category}")

        if self.is_cir_mode:
            grp = item.get("label_group")
            if grp and len(self.group_to_indices[grp]) >= 2:
                pair_idx = self.rng.choice([i for i in self.group_to_indices[grp] if i != idx])
                tar_item = self.samples[pair_idx]
            else:
                tar_item = item
            tar_image = self._load_image(tar_item.get("image_path", ""))
            modifier_text = self._synthesize_cir_modifier(item, tar_item)
            return {
                "ref_image": image, "tar_image": tar_image, "modifier_text": modifier_text,
                "text": text, "label_group": label_group, "product_id": product_id,
                "category": category, "source": source, "semantic_cluster_id": semantic_cluster_id
            }

        return {
            "image": image, "text": text, "label_group": label_group,
            "product_id": product_id, "category": category,
            "source": source, "semantic_cluster_id": semantic_cluster_id
        }


def ecom_collate_fn(batch):
    return {
        "images": [b.get("image", b.get("tar_image")) for b in batch],
        "texts": [b["text"] for b in batch],
        "label_groups": [b["label_group"] for b in batch],
        "product_ids": [b["product_id"] for b in batch],
        "categories": [b["category"] for b in batch],
        "sources": [b.get("source", "unknown") for b in batch],
        "semantic_cluster_ids": [b.get("semantic_cluster_id", "default") for b in batch],
        "ref_images": [b.get("ref_image") for b in batch if "ref_image" in b],
        "modifier_texts": [b.get("modifier_text") for b in batch if "modifier_text" in b]
    }


class AntiOverfitMegaClusterBatchSampler(Sampler):
    """
    Bộ lấy mẫu Batch thông minh chống học vẹt Mega-Cluster (M1 + D4/M3 + M4):
    1. Active Sub-Sampling Rotation (M1): Mỗi epoch chỉ lấy tối đa k=4 biến thể/label_group, xoay vòng qua các epoch.
    2. Smooth Damping Weight (D4 + M3): Cân bằng xác suất lấy mẫu theo sampling_weight = n_src^(-0.5) * n_cluster^(-0.5).
    3. In-Batch Cluster Cap (M4): Tối đa <= 2 mẫu có cùng semantic_cluster_id trong 1 batch để tối đa hóa độ khó Negatives.
    """
    def __init__(
        self,
        samples: list,
        batch_size: int = 256,
        max_active_per_group: int = 4,
        max_per_cluster_in_batch: int = 2,
        seed: int = 42
    ):
        self.samples = samples
        self.batch_size = batch_size
        self.max_active_per_group = max_active_per_group
        self.max_per_cluster_in_batch = max_per_cluster_in_batch
        self.seed = seed
        self.epoch = 0

        self.group_to_indices = defaultdict(list)
        self.index_to_cluster = {}
        self.index_to_weight = np.ones(len(samples), dtype=np.float64)

        for idx, item in enumerate(samples):
            grp = item.get("label_group") or str(item.get("product_id", idx))
            clust = item.get("semantic_cluster_id") or item.get("category", "unknown")
            self.group_to_indices[grp].append(idx)
            self.index_to_cluster[idx] = clust
            self.index_to_weight[idx] = float(item.get("sampling_weight", 1.0))

        est_active = sum(min(len(idxs), self.max_active_per_group) for idxs in self.group_to_indices.values())
        self.num_batches = max(1, est_active // self.batch_size)

    def set_epoch(self, epoch: int):
        self.epoch = epoch

    def __len__(self):
        return self.num_batches

    def __iter__(self):
        rng = np.random.default_rng(self.seed + self.epoch)
        active_indices = []
        for grp, idxs in self.group_to_indices.items():
            k = self.max_active_per_group
            if len(idxs) <= k:
                active_indices.extend(idxs)
            else:
                start = (self.epoch * k) % len(idxs)
                selected = [idxs[(start + offset) % len(idxs)] for offset in range(k)]
                active_indices.extend(selected)

        active_indices = np.array(active_indices, dtype=np.int64)
        weights = self.index_to_weight[active_indices]
        weights = weights / weights.sum()

        ordered = rng.choice(active_indices, size=len(active_indices), replace=False, p=weights).tolist()

        batches = []
        deferred = []
        current_batch = []
        cluster_counts = defaultdict(int)

        for idx in ordered:
            c_id = self.index_to_cluster[idx]
            if cluster_counts[c_id] < self.max_per_cluster_in_batch:
                current_batch.append(idx)
                cluster_counts[c_id] += 1
                if len(current_batch) == self.batch_size:
                    batches.append(current_batch)
                    current_batch = []
                    cluster_counts.clear()
            else:
                deferred.append(idx)

        for idx in deferred:
            current_batch.append(idx)
            if len(current_batch) == self.batch_size:
                batches.append(current_batch)
                current_batch = []

        return iter(batches[:self.num_batches])

In [28]:
# Khởi tạo Dataset và kiểm nghiệm AntiOverfitMegaClusterBatchSampler (Batch Size = 256 chuẩn A100)
train_dataset_example = ECommerceMultimodalDataset(
    jsonl_path=train_clean_file,
    image_root=RAW_IMG_DIR,
    is_train=True,
    seed=42
)

train_sampler_example = AntiOverfitMegaClusterBatchSampler(
    samples=train_dataset_example.samples,
    batch_size=256 if GPU_VRAM_GB >= 35 else 128,
    max_active_per_group=4,
    max_per_cluster_in_batch=2,
    seed=42
)

train_loader_example = DataLoader(
    train_dataset_example,
    batch_sampler=train_sampler_example,
    num_workers=4 if os.name != "nt" else 0,
    pin_memory=True,
    collate_fn=ecom_collate_fn
)

for batch in train_loader_example:
    clust_Counter = Counter(batch["semantic_cluster_ids"])
    print("🎉 Kiểm tra 1 Batch từ AntiOverfitMegaClusterBatchSampler:")
    print(f"   • Kích thước Batch thực tế:        {len(batch['images'])} mẫu")
    print(f"   • Số cụm ngữ nghĩa độc lập:        {len(clust_Counter)} cụm")
    print(f"   • Phân bố nguồn trong Batch:       {dict(Counter(batch['sources']))}")
    print(f"   • Văn bản mẫu [0]:                 {batch['texts'][0]}")
    break

-> Nạp dữ liệu từ: train_clean.jsonl (Chế độ: Train [Multi-Caption + Augment ON])...
   • Đã nạp thành công 153,126 mẫu.
🎉 Kiểm tra 1 Batch từ AntiOverfitMegaClusterBatchSampler:
   • Kích thước Batch thực tế:        256 mẫu
   • Số cụm ngữ nghĩa độc lập:        254 cụm
   • Phân bố nguồn trong Batch:       {'shop100k': 138, 'tiki': 30, 'myntra': 88}
   • Văn bản mẫu [0]:                 Áo sơ mi nam kẻ sọc cổ cài nút tay dài dáng vừa cài cúc


---
## 🧠 Phần 6: Kiến Trúc SigLIP ViT-SO400M & CIR Combiner Module
1. **Backbone SigLIP ViT-SO400M:**
   - Hugging Face: `google/siglip-so400m-patch14-224` (chiều vector đặc trưng $D = 1152$).
   - Sử dụng cơ chế Attention Pooling cho cả nhánh Vision và Text.
2. **Cơ Chế Composed Image Retrieval (CIR Combiner):**
   - Hợp nhất vector ảnh gốc $z_{img}$ và vector câu chỉnh sửa $z_{mod}$:
     $$z_{pred} = \text{LayerNorm}\Big(\text{MLP}([W_{img} z_{img}, W_{txt} z_{mod}]) + \alpha z_{mod} + (1 - \alpha) z_{img}\Big)$$
     trong đó $\alpha = \text{Sigmoid}(W_{gate} [W_{img} z_{img}, W_{txt} z_{mod}]) \in [0, 1]$ là cổng thích ứng động (Dynamic Scalar Gate).
3. **Chiến Lược Tối Ưu Hóa Bộ Nhớ (VRAM T4 15GB):**
   - Bật **Gradient Checkpointing** trên SigLIP.
   - Đóng băng các tầng dưới, mở khóa 4 Blocks Transformer cuối của Vision & Text + LayerNorm + Projection Heads và CIR Combiner.

In [29]:
from transformers import AutoModel, AutoProcessor

class CIRCombiner(nn.Module):
    """
    Mô-đun kết hợp ảnh tham chiếu và câu chỉnh sửa cho Composed Image Retrieval (CIR).
    z_query = f(z_img, z_text_modifier).
    """
    def __init__(self, feature_dim: int = 1152, projection_dim: int = 1024, dropout: float = 0.2):
        super().__init__()
        self.feature_dim = feature_dim
        self.text_proj = nn.Sequential(
            nn.Linear(feature_dim, projection_dim),
            nn.GELU(),
            nn.Dropout(dropout)
        )
        self.image_proj = nn.Sequential(
            nn.Linear(feature_dim, projection_dim),
            nn.GELU(),
            nn.Dropout(dropout)
        )
        self.fusion = nn.Sequential(
            nn.Linear(projection_dim * 2, projection_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(projection_dim, feature_dim)
        )
        # Cổng thích ứng động (Dynamic Scalar Gate)
        self.gate = nn.Sequential(
            nn.Linear(projection_dim * 2, projection_dim // 2),
            nn.GELU(),
            nn.Linear(projection_dim // 2, 1),
            nn.Sigmoid()
        )
        self.norm = nn.LayerNorm(feature_dim)

    def forward(self, img_feat: torch.Tensor, text_feat: torch.Tensor) -> torch.Tensor:
        img_feat = F.normalize(img_feat, p=2, dim=-1)
        text_feat = F.normalize(text_feat, p=2, dim=-1)

        p_img = self.image_proj(img_feat)
        p_txt = self.text_proj(text_feat)
        concat = torch.cat([p_img, p_txt], dim=-1)

        fused = self.fusion(concat)
        alpha = self.gate(concat)

        # Residual connection với trọng số thích ứng động
        out = fused + alpha * text_feat + (1.0 - alpha) * img_feat
        out = self.norm(out)
        return F.normalize(out, p=2, dim=-1)


class MultimodalSigLIPWithCIR(nn.Module):
    """
    Mô hình tích hợp SigLIP Backbone (Vision-Language) và CIR Combiner Module.
    """
    def __init__(
        self,
        model_name: str = "google/siglip-so400m-patch14-224",
        unfreeze_last_blocks: int = 4,
        max_length: int = 64,  # Đã thêm tham số max_length để nhận 64
        device: str = "cuda"
    ):
        super().__init__()
        print(f"-> Đang tải SigLIP Backbone từ: {model_name}...")
        self.device = device
        self.processor = AutoProcessor.from_pretrained(model_name)
        self.siglip = AutoModel.from_pretrained(model_name)

        # Trích xuất kích thước embedding (1152 cho SO400M, 768 cho Base)
        if hasattr(self.siglip.config, "text_config"):
            self.embed_dim = self.siglip.config.text_config.hidden_size
        else:
            self.embed_dim = 1152

        self.max_text_len = max_length

        # Mở rộng Position Embeddings nếu max_length > cấu hình mặc định
        if hasattr(self.siglip, "text_model") and hasattr(self.siglip.text_model, "embeddings"):
            pos_emb = self.siglip.text_model.embeddings.position_embedding
            old_max_len, dim = pos_emb.weight.shape
            if old_max_len < self.max_text_len:
                print(f"⚠️ Mở rộng Text Position Embeddings từ {old_max_len} lên {self.max_text_len} tokens...")
                new_weight = torch.zeros((self.max_text_len, dim), device=pos_emb.weight.device, dtype=pos_emb.weight.dtype)
                new_weight[:old_max_len, :] = pos_emb.weight.data
                # Khởi tạo các token mới bằng cách copy embedding của token cuối cùng
                new_weight[old_max_len:, :] = pos_emb.weight.data[-1:]
                self.siglip.text_model.embeddings.position_embedding = nn.Embedding(self.max_text_len, dim, _weight=new_weight)
                self.siglip.text_model.embeddings.position_ids = torch.arange(self.max_text_len, device=pos_emb.weight.device).expand((1, -1))
                if hasattr(self.siglip.config, "text_config"):
                    self.siglip.config.text_config.max_position_embeddings = self.max_text_len

        print(f"✅ Text Max Position Embeddings: {self.max_text_len}")

        # 1. Kích hoạt Gradient Checkpointing tiết kiệm VRAM
        if hasattr(self.siglip, "gradient_checkpointing_enable"):
            self.siglip.gradient_checkpointing_enable()
            print("✅ Đã bật Gradient Checkpointing.")

        # 2. Khởi tạo CIR Combiner
        self.cir_combiner = CIRCombiner(feature_dim=self.embed_dim, projection_dim=1024)

        # 3. Chiến lược đóng băng / mở khóa tối ưu
        self._setup_unfreezing(unfreeze_last_blocks)

    def _setup_unfreezing(self, unfreeze_last_blocks: int):
        # Đóng băng toàn bộ trước
        for p in self.siglip.parameters():
            p.requires_grad = False

        # Mở khóa các Transformer blocks cuối của Vision Tower
        if hasattr(self.siglip, "vision_model") and hasattr(self.siglip.vision_model, "encoder"):
            for layer in self.siglip.vision_model.encoder.layers[-unfreeze_last_blocks:]:
                for p in layer.parameters():
                    p.requires_grad = True
            if hasattr(self.siglip.vision_model, "post_layernorm"):
                for p in self.siglip.vision_model.post_layernorm.parameters():
                    p.requires_grad = True

        # Mở khóa các Transformer blocks cuối của Text Tower
        if hasattr(self.siglip, "text_model") and hasattr(self.siglip.text_model, "encoder"):
            for layer in self.siglip.text_model.encoder.layers[-unfreeze_last_blocks:]:
                for p in layer.parameters():
                    p.requires_grad = True
            if hasattr(self.siglip.text_model, "final_layer_norm"):
                for p in self.siglip.text_model.final_layer_norm.parameters():
                    p.requires_grad = True

        # CIR Combiner luôn luôn được huấn luyện 100%
        for p in self.cir_combiner.parameters():
            p.requires_grad = True

        trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)
        total = sum(p.numel() for p in self.parameters())
        print(f"📊 Thống kê tham số mô hình:")
        print(f"   • Tổng số tham số:        {total:,}")
        print(f"   • Tham số được train:     {trainable:,} ({trainable/total*100:.2f}%)")
        print(f"   • Tham số đóng băng:      {total - trainable:,} ({(total - trainable)/total*100:.2f}%)")

    def encode_image(self, pixel_values: torch.Tensor) -> torch.Tensor:
        vision_outputs = self.siglip.vision_model(pixel_values=pixel_values)
        img_feats = vision_outputs[1] if len(vision_outputs) > 1 else vision_outputs.pooler_output
        return F.normalize(img_feats, p=2, dim=-1)

    def encode_text(self, input_ids: torch.Tensor, attention_mask: torch.Tensor = None) -> torch.Tensor:
        text_outputs = self.siglip.text_model(input_ids=input_ids, attention_mask=attention_mask)
        txt_feats = text_outputs[1] if len(text_outputs) > 1 else text_outputs.pooler_output
        return F.normalize(txt_feats, p=2, dim=-1)

    def forward(self, pixel_values: torch.Tensor, input_ids: torch.Tensor, attention_mask: torch.Tensor = None):
        img_emb = self.encode_image(pixel_values)
        txt_emb = self.encode_text(input_ids, attention_mask)
        return img_emb, txt_emb

    def compose_query(self, ref_pixels: torch.Tensor, mod_ids: torch.Tensor, mod_mask: torch.Tensor = None):
        ref_img_emb = self.encode_image(ref_pixels)
        mod_txt_emb = self.encode_text(mod_ids, mod_mask)
        return self.cir_combiner(ref_img_emb, mod_txt_emb)

In [30]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoModel, AutoProcessor

class UpgradedAttentionCIRCombiner(nn.Module):
    """
    Mô-đun kết hợp thế hệ mới sử dụng Multi-head Attention (Cross-Attention)
    giúp hòa trộn sâu sắc thông tin ảnh tham chiếu và câu chỉnh sửa.
    """
    def __init__(self, feature_dim: int = 1152, projection_dim: int = 1024, num_heads: int = 8, dropout: float = 0.2):
        super().__init__()
        self.feature_dim = feature_dim
        self.projection_dim = projection_dim

        self.text_proj = nn.Sequential(
            nn.Linear(feature_dim, projection_dim),
            nn.GELU(),
            nn.Dropout(dropout)
        )
        self.image_proj = nn.Sequential(
            nn.Linear(feature_dim, projection_dim),
            nn.GELU(),
            nn.Dropout(dropout)
        )

        # Multi-Head Attention mô hình hóa mối quan hệ chéo (Cross-Attention)
        self.cross_attention = nn.MultiheadAttention(embed_dim=projection_dim, num_heads=num_heads, dropout=dropout, batch_first=True)
        self.norm_att = nn.LayerNorm(projection_dim)

        self.fusion = nn.Sequential(
            nn.Linear(projection_dim, projection_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(projection_dim, feature_dim)
        )

        # Cổng gating động quyết định tỷ lệ đóng góp thông tin
        self.gate = nn.Sequential(
            nn.Linear(projection_dim * 2, projection_dim // 2),
            nn.GELU(),
            nn.Linear(projection_dim // 2, 1),
            nn.Sigmoid()
        )
        self.norm_final = nn.LayerNorm(feature_dim)

    def forward(self, img_feat: torch.Tensor, text_feat: torch.Tensor) -> torch.Tensor:
        img_feat_norm = F.normalize(img_feat, p=2, dim=-1)
        text_feat_norm = F.normalize(text_feat, p=2, dim=-1)

        p_img = self.image_proj(img_feat_norm)
        p_txt = self.text_proj(text_feat_norm)

        # Chuẩn bị sequence chiều dài = 1 phục vụ Attention
        q = p_img.unsqueeze(1)
        k = p_txt.unsqueeze(1)
        v = p_txt.unsqueeze(1)

        att_out, _ = self.cross_attention(q, k, v)
        att_out = att_out.squeeze(1)
        p_combined = self.norm_att(p_img + att_out)

        fused = self.fusion(p_combined)

        concat_gate = torch.cat([p_img, p_txt], dim=-1)
        alpha = self.gate(concat_gate)

        out = fused + alpha * text_feat_norm + (1.0 - alpha) * img_feat_norm
        out = self.norm_final(out)
        return F.normalize(out, p=2, dim=-1)

class HighlyEffectiveSigLIP(nn.Module):
    """
    Mô hình SigLIP tối ưu hóa cao độ, unfreeze sâu 8 blocks và tích hợp combiner thông minh.
    """
    def __init__(
        self,
        model_name: str = "google/siglip-so400m-patch14-224",
        unfreeze_last_blocks: int = 8,
        max_length: int = 128,
        device: str = "cuda"
    ):
        super().__init__()
        print(f"[Upgrade] Đang khởi tạo siêu mô hình từ checkpoint: {model_name}...")
        self.device = device
        self.processor = AutoProcessor.from_pretrained(model_name)
        self.siglip = AutoModel.from_pretrained(model_name)

        if hasattr(self.siglip.config, "text_config"):
            self.embed_dim = self.siglip.config.text_config.hidden_size
        else:
            self.embed_dim = 1152

        self.max_text_len = max_length
        self.cir_combiner = UpgradedAttentionCIRCombiner(feature_dim=self.embed_dim, projection_dim=1024)

        if hasattr(self.siglip, "gradient_checkpointing_enable"):
            self.siglip.gradient_checkpointing_enable()
            print("⚡ Đã kích hoạt Gradient Checkpointing giúp tối ưu hóa dung lượng VRAM.")

        self._setup_unfreezing(unfreeze_last_blocks)

    def _setup_unfreezing(self, unfreeze_last_blocks: int):
        # Đóng băng toàn bộ mạng trước
        for p in self.siglip.parameters():
            p.requires_grad = False

        # Unfreeze sâu 8 blocks cuối cùng để bứt phá mAP
        if hasattr(self.siglip, "vision_model") and hasattr(self.siglip.vision_model, "encoder"):
            for layer in self.siglip.vision_model.encoder.layers[-unfreeze_last_blocks:]:
                for p in layer.parameters():
                    p.requires_grad = True
            if hasattr(self.siglip.vision_model, "post_layernorm"):
                for p in self.siglip.vision_model.post_layernorm.parameters():
                    p.requires_grad = True

        if hasattr(self.siglip, "text_model") and hasattr(self.siglip.text_model, "encoder"):
            for layer in self.siglip.text_model.encoder.layers[-unfreeze_last_blocks:]:
                for p in layer.parameters():
                    p.requires_grad = True
            if hasattr(self.siglip.text_model, "final_layer_norm"):
                for p in self.siglip.text_model.final_layer_norm.parameters():
                    p.requires_grad = True

        for p in self.cir_combiner.parameters():
            p.requires_grad = True

        trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)
        total = sum(p.numel() for p in self.parameters())
        print(f"📊 [Upgraded Model] Tham số huấn luyện được: {trainable:,} ({trainable/total*100:.2f}%)")

    def encode_image(self, pixel_values: torch.Tensor) -> torch.Tensor:
        vision_outputs = self.siglip.vision_model(pixel_values=pixel_values)
        img_feats = vision_outputs[1] if len(vision_outputs) > 1 else vision_outputs.pooler_output
        return F.normalize(img_feats, p=2, dim=-1)

    def encode_text(self, input_ids: torch.Tensor, attention_mask: torch.Tensor = None) -> torch.Tensor:
        text_outputs = self.siglip.text_model(input_ids=input_ids, attention_mask=attention_mask)
        txt_feats = text_outputs[1] if len(text_outputs) > 1 else text_outputs.pooler_output
        return F.normalize(txt_feats, p=2, dim=-1)

    def forward(self, pixel_values: torch.Tensor, input_ids: torch.Tensor, attention_mask: torch.Tensor = None):
        img_emb = self.encode_image(pixel_values)
        txt_emb = self.encode_text(input_ids, attention_mask)
        return img_emb, txt_emb

    def compose_query(self, ref_pixels: torch.Tensor, mod_ids: torch.Tensor, mod_mask: torch.Tensor = None):
        ref_img_emb = self.encode_image(ref_pixels)
        mod_txt_emb = self.encode_text(mod_ids, mod_mask)
        return self.cir_combiner(ref_img_emb, mod_txt_emb)


### 🔄 Cấu Hình Khởi Tạo Model Khác Để Đột Phá Hiệu Năng
Dưới đây là cách bạn chuyển đổi sang các cấu hình backbone khác như bản SigLIP đa ngôn ngữ hoặc bản độ phân giải cao ($384\times384$) để cải thiện khả năng hội tụ ngữ nghĩa tiếng Việt.

In [31]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoModel, AutoProcessor

# 1. Định nghĩa bộ kết hợp đặc trưng nâng cao sử dụng Multi-head Attention
class UpgradedAttentionCIRCombiner(nn.Module):
    def __init__(self, feature_dim: int = 1152, projection_dim: int = 1024, num_heads: int = 8, dropout: float = 0.2):
        super().__init__()
        self.feature_dim = feature_dim
        self.projection_dim = projection_dim
        self.text_proj = nn.Sequential(
            nn.Linear(feature_dim, projection_dim),
            nn.GELU(),
            nn.Dropout(dropout)
        )
        self.image_proj = nn.Sequential(
            nn.Linear(feature_dim, projection_dim),
            nn.GELU(),
            nn.Dropout(dropout)
        )
        self.cross_attention = nn.MultiheadAttention(embed_dim=projection_dim, num_heads=num_heads, dropout=dropout, batch_first=True)
        self.norm_att = nn.LayerNorm(projection_dim)
        self.fusion = nn.Sequential(
            nn.Linear(projection_dim, projection_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(projection_dim, feature_dim)
        )
        self.gate = nn.Sequential(
            nn.Linear(projection_dim * 2, projection_dim // 2),
            nn.GELU(),
            nn.Linear(projection_dim // 2, 1),
            nn.Sigmoid()
        )
        self.norm_final = nn.LayerNorm(feature_dim)

    def forward(self, img_feat: torch.Tensor, text_feat: torch.Tensor) -> torch.Tensor:
        img_feat_norm = F.normalize(img_feat, p=2, dim=-1)
        text_feat_norm = F.normalize(text_feat, p=2, dim=-1)
        p_img = self.image_proj(img_feat_norm)
        p_txt = self.text_proj(text_feat_norm)
        q = p_img.unsqueeze(1)
        k = p_txt.unsqueeze(1)
        v = p_txt.unsqueeze(1)
        att_out, _ = self.cross_attention(q, k, v)
        att_out = att_out.squeeze(1)
        p_combined = self.norm_att(p_img + att_out)
        fused = self.fusion(p_combined)
        concat_gate = torch.cat([p_img, p_txt], dim=-1)
        alpha = self.gate(concat_gate)
        out = fused + alpha * text_feat_norm + (1.0 - alpha) * img_feat_norm
        out = self.norm_final(out)
        return F.normalize(out, p=2, dim=-1)

# 2. Định nghĩa cấu trúc lớp HighlyEffectiveSigLIP độc lập
class HighlyEffectiveSigLIP(nn.Module):
    def __init__(self, model_name: str = "google/siglip-base-patch16-256-multilingual", unfreeze_last_blocks: int = 8, max_length: int = 128, device: str = "cuda"):
        super().__init__()
        print(f"[Upgrade] Đang khởi tạo mô hình từ: {model_name}...")
        self.device = device
        self.processor = AutoProcessor.from_pretrained(model_name)
        self.siglip = AutoModel.from_pretrained(model_name)
        if hasattr(self.siglip.config, "text_config"):
            self.embed_dim = self.siglip.config.text_config.hidden_size
        else:
            self.embed_dim = 768 # Mặc định cho bản Base
        self.max_text_len = max_length
        self.cir_combiner = UpgradedAttentionCIRCombiner(feature_dim=self.embed_dim, projection_dim=1024)
        if hasattr(self.siglip, "gradient_checkpointing_enable"):
            self.siglip.gradient_checkpointing_enable()
            print("⚡ Đã kích hoạt Gradient Checkpointing để tiết kiệm VRAM.")
        self._setup_unfreezing(unfreeze_last_blocks)

    def _setup_unfreezing(self, unfreeze_last_blocks: int):
        for p in self.siglip.parameters():
            p.requires_grad = False
        if hasattr(self.siglip, "vision_model") and hasattr(self.siglip.vision_model, "encoder"):
            for layer in self.siglip.vision_model.encoder.layers[-unfreeze_last_blocks:]:
                for p in layer.parameters():
                    p.requires_grad = True
        if hasattr(self.siglip, "text_model") and hasattr(self.siglip.text_model, "encoder"):
            for layer in self.siglip.text_model.encoder.layers[-unfreeze_last_blocks:]:
                for p in layer.parameters():
                    p.requires_grad = True
        for p in self.cir_combiner.parameters():
            p.requires_grad = True
        trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)
        total = sum(p.numel() for p in self.parameters())
        print(f"📊 Tham số huấn luyện được: {trainable:,} ({trainable/total*100:.2f}%)")

# 3. Khởi tạo thực tế với Backbone Đa Ngôn Ngữ
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
ALTERNATIVE_MODEL_NAME = "google/siglip-base-patch16-256-multilingual"

print(f"-> Bắt đầu tiến trình với: {ALTERNATIVE_MODEL_NAME}")
try:
    model = HighlyEffectiveSigLIP(
        model_name=ALTERNATIVE_MODEL_NAME,
        unfreeze_last_blocks=8,
        max_length=128,
        device=DEVICE
    ).to(DEVICE)
    print("✅ Khởi tạo cấu hình mô hình mới thành công! Sẵn sàng nạp vào Trainer.")
except Exception as e:
    print(f"❌ Lỗi khi tải mô hình: {e}")

-> Bắt đầu tiến trình với: google/siglip-base-patch16-256-multilingual
[Upgrade] Đang khởi tạo mô hình từ: google/siglip-base-patch16-256-multilingual...


Loading weights:   0%|          | 0/408 [00:00<?, ?it/s]

⚡ Đã kích hoạt Gradient Checkpointing để tiết kiệm VRAM.
📊 Tham số huấn luyện được: 122,069,249 (32.18%)
✅ Khởi tạo cấu hình mô hình mới thành công! Sẵn sàng nạp vào Trainer.


---
## 🎯 Phần 7: Hàm Mất Mát SigLIP Sigmoid Loss Với 3-Tier False-Negative Masking & Vectorized Multi-Similarity Loss
1. **SigLIP Pairwise Sigmoid Loss (Zhai et al., 2023):**
   $$\mathcal{L} = -\frac{1}{B} \sum_{i=1}^B \sum_{j=1}^B M_{ij} \log \sigma\Big(y_{ij} (t \cdot z_i^T z_j + b)\Big)$$
2. **3-Tier False-Negative Masking ($M_{ij} = 0$ với các cặp ngoài đường chéo thỏa 1 trong 3 điều kiện):**
   - **Tier 1 (`same_label_group`):** Hai mẫu thuộc cùng nhóm sản phẩm `label_group`.
   - **Tier 2 (`same_normalized_text`):** Hai mẫu có câu văn bản mô tả chuẩn hóa trùng khớp nhau.
   - **Tier 3 (`semantic_near_identical`):** Độ tương đồng ngữ nghĩa trong batch $\cos(z^{txt}_i, z^{txt}_j) > 0.95$ (ngăn chặn phạt oan các mẫu áo/sản phẩm giống hệt nhau từ khác nguồn).
3. **Vectorized Hard-Negative Mining:** Tính toán hoàn toàn bằng ma trận PyTorch trên GPU A100 (loại bỏ hoàn toàn vòng lặp `for i in range(B)` trên CPU).

In [32]:
import math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

class SigLIPLoss(nn.Module):
    """
    Hàm mất mát SigLIP Sigmoid Loss với Learnable Temperature và Bias.
    Tích hợp 3-Tier False-Negative Masking:
      1. Cùng label_group
      2. Cùng câu văn bản sau chuẩn hóa (normalized text)
      3. Độ tương đồng vector văn bản trong batch cos(txt_i, txt_j) > semantic_mask_thresh (0.95)
    """
    def __init__(
        self,
        init_temp: float = 10.0,
        init_bias: float = -10.0,
        semantic_mask_thresh: float = 0.95
    ):
        super().__init__()
        self.t_prime = nn.Parameter(torch.tensor(math.log(init_temp), dtype=torch.float32))
        self.bias = nn.Parameter(torch.tensor(init_bias, dtype=torch.float32))
        self.semantic_mask_thresh = semantic_mask_thresh

    def forward(
        self,
        text_embeds: torch.Tensor,
        image_embeds: torch.Tensor,
        label_groups: list = None,
        texts: list = None
    ) -> torch.Tensor:
        text_embeds = F.normalize(text_embeds.float(), p=2, dim=-1)
        image_embeds = F.normalize(image_embeds.float(), p=2, dim=-1)

        scale = torch.clamp(torch.exp(self.t_prime), max=100.0)
        logits = scale * torch.matmul(text_embeds, image_embeds.t()) + self.bias
        B = logits.size(0)
        device = logits.device

        # Đường chéo chính là +1, ngoài đường chéo là -1
        eye_mask = torch.eye(B, device=device, dtype=torch.bool)
        labels = torch.where(eye_mask, 1.0, -1.0)

        false_neg_mask = torch.zeros((B, B), device=device, dtype=torch.bool)

        # Tier 1: Cùng label_group
        if label_groups is not None:
            grp_arr = np.array(label_groups)
            same_grp = torch.tensor(grp_arr[:, None] == grp_arr[None, :], device=device, dtype=torch.bool)
            false_neg_mask = false_neg_mask | same_grp

        # Tier 2: Cùng chuỗi văn bản chuẩn hóa
        if texts is not None:
            norm_txts = np.array([" ".join(str(t).lower().split()) for t in texts])
            same_txt = torch.tensor(norm_txts[:, None] == norm_txts[None, :], device=device, dtype=torch.bool)
            false_neg_mask = false_neg_mask | same_txt

        # Tier 3: Tương đồng ngữ nghĩa rất cao cos(txt_i, txt_j) > 0.95
        if self.semantic_mask_thresh is not None and self.semantic_mask_thresh < 1.0:
            with torch.no_grad():
                txt_sim = torch.matmul(text_embeds.detach(), text_embeds.detach().t())
                high_txt_sim = txt_sim > self.semantic_mask_thresh
            false_neg_mask = false_neg_mask | high_txt_sim

        # Chỉ che các cặp ngoài đường chéo (giữ nguyên đường chéo chính làm positive)
        mask_out = false_neg_mask & (~eye_mask)

        loss_matrix = F.softplus(-labels * logits)
        loss_matrix = loss_matrix.masked_fill(mask_out, 0.0)

        valid_per_row = (~mask_out).float().sum(dim=1).clamp_min(1.0)
        row_loss = (loss_matrix.sum(dim=1) / valid_per_row) * (B / valid_per_row.mean())
        return row_loss.mean() / B * B


class LearnableInfoNCELoss(nn.Module):
    """NT-Xent / InfoNCE Loss đối xứng với learnable temperature."""
    def __init__(self, init_temp: float = 0.07):
        super().__init__()
        self.logit_scale = nn.Parameter(torch.tensor(math.log(1.0 / init_temp), dtype=torch.float32))

    def forward(self, q_embeds: torch.Tensor, k_embeds: torch.Tensor) -> torch.Tensor:
        scale = torch.clamp(self.logit_scale.exp(), max=100.0)
        sim = scale * torch.matmul(q_embeds.float(), k_embeds.float().t())
        B = sim.size(0)
        targets = torch.arange(B, device=sim.device)
        return 0.5 * (F.cross_entropy(sim, targets) + F.cross_entropy(sim.t(), targets))

In [33]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

class EnhancedECommerceLoss(nn.Module):
    """
    Hàm Loss cải tiến Vectorized hoàn toàn trên GPU A100:
    1. SigLIP Pairwise Sigmoid Loss với 3-Tier False-Negative Masking.
    2. Vectorized Soft-Margin Multi-Negative Mining (không dùng vòng lặp for).
    """
    def __init__(
        self,
        init_temp: float = 10.0,
        init_bias: float = -10.0,
        triplet_margin: float = 0.25,
        weight_triplet: float = 0.5,
        dropout_rate: float = 0.1,
        semantic_mask_thresh: float = 0.95
    ):
        super().__init__()
        self.siglip_loss_fn = SigLIPLoss(
            init_temp=init_temp,
            init_bias=init_bias,
            semantic_mask_thresh=semantic_mask_thresh
        )
        self.triplet_margin = triplet_margin
        self.weight_triplet = weight_triplet
        self.dropout = nn.Dropout(p=dropout_rate)
        self.semantic_mask_thresh = semantic_mask_thresh

    def forward(
        self,
        text_embeds: torch.Tensor,
        image_embeds: torch.Tensor,
        label_groups: list,
        categories: list = None,
        texts: list = None
    ) -> torch.Tensor:
        text_embeds = F.normalize(self.dropout(text_embeds.float()), p=2, dim=-1)
        image_embeds = F.normalize(self.dropout(image_embeds.float()), p=2, dim=-1)

        base_loss = self.siglip_loss_fn(text_embeds, image_embeds, label_groups, texts=texts)

        sim = torch.matmul(text_embeds, image_embeds.t())
        dist = 1.0 - sim
        B = dist.size(0)
        device = dist.device
        eye_mask = torch.eye(B, device=device, dtype=torch.bool)

        grp_arr = np.array(label_groups)
        is_pos = torch.tensor(grp_arr[:, None] == grp_arr[None, :], device=device, dtype=torch.bool)

        false_neg = is_pos.clone()
        if texts is not None:
            norm_txts = np.array([" ".join(str(t).lower().split()) for t in texts])
            false_neg = false_neg | torch.tensor(norm_txts[:, None] == norm_txts[None, :], device=device, dtype=torch.bool)

        with torch.no_grad():
            high_txt_sim = torch.matmul(text_embeds.detach(), text_embeds.detach().t()) > self.semantic_mask_thresh
        false_neg = false_neg | high_txt_sim

        is_neg = (~false_neg) & (~eye_mask)

        # Khoảng cách positive lớn nhất trên từng hàng (kể cả đường chéo chính)
        pos_dists = dist.masked_fill(~is_pos, -1.0)
        max_pos = pos_dists.max(dim=1, keepdim=True).values.clamp_min(0.0)

        violations = (max_pos - dist + self.triplet_margin).masked_fill(~is_neg, -1e4)
        active_any = (violations > 0.0).any(dim=1)
        if active_any.any():
            row_soft = torch.log1p(torch.exp(torch.clamp(violations, max=20.0)) * (violations > 0.0).float()).sum(dim=1)
            triplet_loss = row_soft[active_any].mean()
        else:
            triplet_loss = torch.tensor(0.0, device=device)

        return base_loss + self.weight_triplet * triplet_loss

### 🔄 Cập Nhật Trainer Để Sử Dụng Hàm Loss Mới Nâng Cao
Dưới đây là tế bào cấu hình lại Loss Function sang `EnhancedECommerceLoss` giúp đẩy mạnh mAP lên mức tối đa nhờ cơ chế đào bới mẫu khó (Hard Negative Mining).

In [34]:
# Thay thế loss_fn bằng EnhancedECommerceLoss để tối ưu hóa mAP tối đa
loss_fn = EnhancedECommerceLoss(
    init_temp=10.0,
    init_bias=-10.0,
    triplet_margin=0.25, # Tăng biên độ phân tách giữa sản phẩm đúng và sản phẩm gây nhầm lẫn
    weight_triplet=0.5   # Trọng số cho cơ chế tối ưu mAP
)

print("✅ Đã thiết lập EnhancedECommerceLoss với cơ chế Hard-Negative Triplet Mining!")

✅ Đã thiết lập EnhancedECommerceLoss với cơ chế Hard-Negative Triplet Mining!


---
## 📏 Phần 8: Bộ Đánh Giá Tra Cứu Toàn Diện (Recall@1, Recall@5, Recall@10 & mAP)
Đánh giá năng lực truy hồi sản phẩm chuẩn E-commerce:
- **Recall@K (K = 1, 5, 10):** Tỷ lệ phần trăm truy vấn tìm được ít nhất 1 sản phẩm đúng cụm trong Top-K đề xuất.
- **Mean Average Precision (mAP):** Đo lường độ chính xác trung bình toàn bộ vị trí xếp hạng của các kết quả liên quan.

In [35]:
import numpy as np
import torch
import torch.nn.functional as F

@torch.no_grad()
def compute_retrieval_metrics(
    query_embeds: torch.Tensor,
    gallery_embeds: torch.Tensor,
    query_groups: list,
    gallery_groups: list,
    top_k_list: list = [1, 5, 10],
    query_sources: list = None
) -> dict:
    """
    Tính toán nhanh Recall@1, Recall@5, Recall@10 và mAP bằng ma trận GPU PyTorch.
    Đồng thời báo cáo chỉ số phân tách theo miền (Fashion vs Non-Fashion) theo thiết kế D5.
    """
    query_embeds = F.normalize(query_embeds.float(), p=2, dim=-1)
    gallery_embeds = F.normalize(gallery_embeds.float(), p=2, dim=-1)
    sim_matrix = torch.matmul(query_embeds, gallery_embeds.t())
    N = sim_matrix.size(0)

    max_k = max(top_k_list)
    _, top_indices = sim_matrix.topk(min(max_k, gallery_embeds.size(0)), dim=1)
    top_indices = top_indices.cpu().numpy()

    query_groups = np.array(query_groups)
    gallery_groups = np.array(gallery_groups)

    hits = {k: np.zeros(N, dtype=np.float32) for k in top_k_list}
    aps = np.zeros(N, dtype=np.float32)

    for i in range(N):
        q_grp = query_groups[i]
        retrieved_grps = gallery_groups[top_indices[i]]

        for k in top_k_list:
            if q_grp in retrieved_grps[:k]:
                hits[k][i] = 1.0

        matches = (retrieved_grps == q_grp)
        if np.any(matches):
            ranks = np.where(matches)[0] + 1
            aps[i] = np.mean(np.arange(1, len(ranks) + 1) / ranks)

    metrics = {f"Recall@{k}": float(np.mean(hits[k]) * 100.0) for k in top_k_list}
    metrics["mAP"] = float(np.mean(aps))

    if query_sources is not None and len(query_sources) == N:
        srcs = np.array(query_sources)
        fashion_mask = np.isin(srcs, ["myntra", "s100k", "shop100k"])
        non_fashion_mask = (srcs == "tiki")
        if np.any(fashion_mask):
            metrics["Fashion_Recall@10"] = float(np.mean(hits[10][fashion_mask]) * 100.0)
            metrics["Fashion_mAP"] = float(np.mean(aps[fashion_mask]))
        if np.any(non_fashion_mask):
            metrics["NonFashion_Recall@10"] = float(np.mean(hits[10][non_fashion_mask]) * 100.0)
            metrics["NonFashion_mAP"] = float(np.mean(aps[non_fashion_mask]))

    return metrics

In [36]:
import torch
import torch.nn as nn

def expand_siglip_position_embeddings(model, target_max_length=128):
    """
    Mở rộng ma trận nhúng vị trí của SigLIP Text Model từ 64 lên 128 tokens
    để tương thích hoàn toàn với tham số tiền xử lý max_length = 128.
    """
    if not hasattr(model, "siglip") or not hasattr(model.siglip, "text_model"):
        print("⚠️ Không tìm thấy cấu trúc text_model trong đối tượng model.")
        return

    text_model = model.siglip.text_model
    pos_emb = text_model.embeddings.position_embedding
    old_max_len, dim = pos_emb.weight.shape

    if old_max_len >= target_max_length:
        print(f"✅ Ma trận nhúng vị trí đã có kích thước lớn hơn hoặc bằng mục tiêu: {old_max_len} >= {target_max_length}")
        return

    print(f"⚠️ Đang mở rộng Text Position Embeddings từ {old_max_len} lên {target_max_length} tokens...")

    # 1. Tạo ma trận trọng số mới
    new_weight = torch.zeros((target_max_length, dim), device=pos_emb.weight.device, dtype=pos_emb.weight.dtype)

    # 2. Sao chép trọng số cũ (vị trí từ 0 đến 63)
    new_weight[:old_max_len, :] = pos_emb.weight.data

    # 3. Khởi tạo các vị trí mới (64 đến 127) bằng cách lặp lại embedding của vị trí cuối cùng
    new_weight[old_max_len:, :] = pos_emb.weight.data[-1:]

    # 4. Gán lại nn.Embedding mới vào mô hình
    text_model.embeddings.position_embedding = nn.Embedding(target_max_length, dim, _weight=new_weight)

    # 5. Cập nhật lại dải position_ids tương ứng
    text_model.embeddings.position_ids = torch.arange(target_max_length, device=pos_emb.weight.device).expand((1, -1))

    # 6. Cập nhật lại config
    if hasattr(model.siglip.config, "text_config"):
        model.siglip.config.text_config.max_position_embeddings = target_max_length
    model.max_text_len = target_max_length

    print(f"✅ Đã mở rộng thành công lên {target_max_length}! Sẵn sàng huấn luyện với max_length = 128.")

# Áp dụng trực tiếp lên đối tượng model hiện tại trong session
if 'model' in globals():
    expand_siglip_position_embeddings(model, target_max_length=128)

⚠️ Đang mở rộng Text Position Embeddings từ 64 lên 128 tokens...
✅ Đã mở rộng thành công lên 128! Sẵn sàng huấn luyện với max_length = 128.


---
## ⚙️ Phần 9: Huấn Luyện Viên (Trainer) & Chiến Lược Early Stopping (Patience = 3)
- **Tối ưu hóa VRAM & Mini-Batch:**
  - Mini-batch: **32 hoặc 64**.
  - **Bảo toàn Effective Batch Size = 256**:
    $$\text{gradient\_accumulation\_steps} = \max\left(1, \frac{256}{\text{mini\_batch\_size} \times \text{num\_gpus}}\right)$$
  - Mixed Precision `autocast(dtype=torch.float16)` + `GradScaler()`.
- **Optimizer & Scheduler:**
  - AdamW (`lr=1e-4`, `weight_decay=1e-2`).
  - CosineAnnealingWarmup (1000 warmup steps).
- **Early Stopping (Patience = 3 epochs):**
  - Giám sát chỉ số `Val Loss`. Nếu 3 epoch liên tiếp không cải thiện, ngắt quá trình huấn luyện và thông báo `EarlyStopping triggered`.
  - Tự động lưu checkpoint tốt nhất vào `best_model.pt` và đồng bộ vào Google Drive.

In [37]:
from transformers import get_cosine_schedule_with_warmup
import json
import os
import shutil
import time
import gc
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

class EarlyStopping:
    """Early Stopping theo dõi Val Loss với patience = 5 epochs."""
    def __init__(self, patience: int = 5, delta: float = 1e-4, checkpoint_path: str = "best_model.pt"):
        self.patience = patience
        self.delta = delta
        self.checkpoint_path = checkpoint_path
        self.counter = 0
        self.best_loss = float('inf')
        self.early_stop = False
        self.best_epoch = -1

    def step(self, val_loss: float, epoch: int, model: nn.Module, backup_dir: str = None) -> bool:
        if val_loss < self.best_loss - self.delta:
            self.best_loss = val_loss
            self.best_epoch = epoch
            self.counter = 0
            self.save_checkpoint(model, backup_dir)
            return True
        else:
            self.counter += 1
            print(f"   [EarlyStopping] Val Loss không cải thiện. Counter: {self.counter}/{self.patience}")
            if self.counter >= self.patience:
                self.early_stop = True
                print(f"⌛ EarlyStopping triggered! Dừng quá trình huấn luyện tại Epoch {epoch}.")
            return False

    def save_checkpoint(self, model: nn.Module, backup_dir: str = None):
        torch.save(model.state_dict(), self.checkpoint_path)
        if backup_dir and os.path.exists(backup_dir):
            shutil.copy2(self.checkpoint_path, os.path.join(backup_dir, "best_model.pt"))
        print(f"   💾 Đã lưu checkpoint tốt nhất tại: {self.checkpoint_path}")


class MultimodalTrainer:
    """
    Bộ huấn luyện SigLIP & CIR tối ưu hóa cho GPU A100:
    - Tự động dùng Native BFloat16 trên A100 (không cần GradScaler tránh nghẽn đồng bộ GPU-CPU).
    - Tự động xoay vòng biến thể (set_epoch) trên AntiOverfitMegaClusterBatchSampler.
    - Truyền danh sách texts vào hàm Loss để kích hoạt 3-Tier False-Negative Masking.
    """
    def __init__(
        self, model, train_loader, val_loader, test_loader, loss_fn,
        device="cuda", batch_size=256, learning_rate=1e-4, weight_decay=1e-2,
        max_epochs=10, warmup_steps=1000, checkpoint_dir="checkpoints",
        drive_backup_dir=None, patience=5
    ):
        self.model = model.to(device)
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.test_loader = test_loader
        self.loss_fn = loss_fn.to(device) if isinstance(loss_fn, nn.Module) else loss_fn
        self.device = device
        self.batch_size = batch_size
        self.max_epochs = max_epochs
        self.checkpoint_dir = checkpoint_dir
        self.drive_backup_dir = drive_backup_dir

        # Nhận diện BFloat16 trên GPU A100 / Ampere+
        self.is_ampere = torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8
        self.amp_dtype = torch.bfloat16 if self.is_ampere else torch.float16
        self.use_scaler = not self.is_ampere

        os.makedirs(checkpoint_dir, exist_ok=True)
        self.best_checkpoint_path = os.path.join(checkpoint_dir, "best_model.pt")
        self.last_checkpoint_path = os.path.join(checkpoint_dir, "last_model.pt")
        self.history_path = os.path.join(checkpoint_dir, "train_history.json")

        self.early_stopping = EarlyStopping(patience=patience, checkpoint_path=self.best_checkpoint_path)

        backbone_params = [p for name, p in self.model.named_parameters() if "siglip" in name and p.requires_grad]
        head_params = [p for name, p in self.model.named_parameters() if "siglip" not in name and p.requires_grad]
        if isinstance(self.loss_fn, nn.Module):
            head_params.extend([p for p in self.loss_fn.parameters() if p.requires_grad])

        self.optimizer = torch.optim.AdamW([
            {"params": backbone_params, "lr": learning_rate * 0.1},
            {"params": head_params, "lr": learning_rate}
        ], weight_decay=weight_decay, fused=torch.cuda.is_available())

        total_steps = max(1, len(self.train_loader) * max_epochs)
        self.scheduler = get_cosine_schedule_with_warmup(self.optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps)
        self.scaler = torch.amp.GradScaler('cuda', enabled=self.use_scaler)

        self.train_history = {
            "epoch": [], "train_loss": [], "val_loss": [],
            "val_r1": [], "val_r5": [], "val_r10": [],
            "val_map": [], "epoch_time": []
        }

    def train_epoch(self, epoch: int) -> float:
        self.model.train()
        if hasattr(self.train_loader, "batch_sampler") and hasattr(self.train_loader.batch_sampler, "set_epoch"):
            self.train_loader.batch_sampler.set_epoch(epoch)

        total_loss = 0.0
        valid_steps = 0
        pbar = tqdm(self.train_loader, desc=f"Epoch {epoch}/{self.max_epochs} [Train {'BF16-A100' if self.is_ampere else 'FP16'}]")
        processor = self.model.processor

        for step, batch in enumerate(pbar):
            self.optimizer.zero_grad(set_to_none=True)

            inputs = processor(
                images=batch["images"],
                text=batch["texts"],
                padding="max_length",
                truncation=True,
                max_length=self.model.max_text_len,
                return_tensors="pt"
            )
            pixel_values = inputs["pixel_values"].to(self.device, non_blocking=True)
            input_ids = inputs["input_ids"].to(self.device, non_blocking=True)
            attention_mask = inputs.get("attention_mask").to(self.device, non_blocking=True) if "attention_mask" in inputs else None

            with torch.amp.autocast(device_type='cuda', dtype=self.amp_dtype):
                img_emb, txt_emb = self.model(pixel_values, input_ids, attention_mask)
                try:
                    loss = self.loss_fn(txt_emb, img_emb, batch["label_groups"], batch.get("categories"), texts=batch["texts"])
                except TypeError:
                    loss = self.loss_fn(txt_emb, img_emb, batch["label_groups"], batch.get("categories"))

            if torch.isnan(loss) or torch.isinf(loss):
                continue

            if self.use_scaler:
                self.scaler.scale(loss).backward()
                self.scaler.unscale_(self.optimizer)
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)
                old_scale = self.scaler.get_scale()
                self.scaler.step(self.optimizer)
                self.scaler.update()
                if self.scaler.get_scale() >= old_scale:
                    self.scheduler.step()
            else:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)
                self.optimizer.step()
                self.scheduler.step()

            total_loss += loss.item()
            valid_steps += 1
            pbar.set_postfix({"Loss": f"{loss.item():.4f}", "LR": f"{self.optimizer.param_groups[1]['lr']:.2e}"})

        return total_loss / max(1, valid_steps)

    @torch.no_grad()
    def evaluate(self, dataloader, desc="Evaluation") -> dict:
        self.model.eval()
        total_loss = 0.0
        all_img_embs, all_txt_embs = [], []
        all_groups, all_sources = [], []
        processor = self.model.processor

        for batch in tqdm(dataloader, desc=desc):
            inputs = processor(
                images=batch["images"],
                text=batch["texts"],
                padding="max_length",
                truncation=True,
                max_length=self.model.max_text_len,
                return_tensors="pt"
            )
            pixel_values = inputs["pixel_values"].to(self.device, non_blocking=True)
            input_ids = inputs["input_ids"].to(self.device, non_blocking=True)
            attention_mask = inputs.get("attention_mask").to(self.device, non_blocking=True) if "attention_mask" in inputs else None

            with torch.amp.autocast(device_type='cuda', dtype=self.amp_dtype):
                img_emb, txt_emb = self.model(pixel_values, input_ids, attention_mask)
                try:
                    loss = self.loss_fn(txt_emb, img_emb, batch["label_groups"], batch.get("categories"), texts=batch["texts"])
                except TypeError:
                    loss = self.loss_fn(txt_emb, img_emb, batch["label_groups"], batch.get("categories"))

            total_loss += loss.item()
            all_img_embs.append(img_emb.float().cpu())
            all_txt_embs.append(txt_emb.float().cpu())
            all_groups.extend(batch["label_groups"])
            all_sources.extend(batch.get("sources", ["unknown"] * len(batch["label_groups"])))

        img_embs = torch.cat(all_img_embs, dim=0)
        txt_embs = torch.cat(all_txt_embs, dim=0)

        metrics = compute_retrieval_metrics(txt_embs, img_embs, all_groups, all_groups, query_sources=all_sources)
        metrics["loss"] = total_loss / max(1, len(dataloader))
        return metrics

    def save_history(self):
        history_data = self.train_history
        if os.path.exists(self.history_path):
            try:
                with open(self.history_path, "r", encoding="utf-8") as f:
                    existing_history = json.load(f)
                merged_history = {k: list(existing_history.get(k, [])) for k in self.train_history.keys()}
                for idx, epoch in enumerate(self.train_history["epoch"]):
                    if epoch not in merged_history["epoch"]:
                        for k in self.train_history.keys():
                            merged_history[k].append(self.train_history[k][idx])
                history_data = merged_history
            except Exception:
                pass

        with open(self.history_path, "w", encoding="utf-8") as f:
            json.dump(history_data, f, indent=2, ensure_ascii=False)

        if self.drive_backup_dir and os.path.exists(self.drive_backup_dir):
            drive_path = os.path.join(self.drive_backup_dir, "train_history.json")
            with open(drive_path, "w", encoding="utf-8") as f:
                json.dump(history_data, f, indent=2, ensure_ascii=False)

    def evaluate_test(self, golden_jsonl_path: str = None) -> dict:
        if os.path.exists(self.best_checkpoint_path):
            print(f"-> Nạp trọng số tối ưu từ: {self.best_checkpoint_path}")
            self.model.load_state_dict(torch.load(self.best_checkpoint_path, map_location=self.device))
        metrics = self.evaluate(self.test_loader, desc="Evaluation [Test]")

        # Đánh giá thêm trên bộ 400 truy vấn thực tế (Golden Evaluation Set) nếu có
        if golden_jsonl_path and os.path.exists(golden_jsonl_path):
            golden_qs, golden_grps = [], []
            with open(golden_jsonl_path, "r", encoding="utf-8") as f:
                for line in f:
                    if line.strip():
                        obj = json.loads(line)
                        golden_qs.append(obj["query_text"])
                        golden_grps.append(obj["label_group"])
            if golden_qs:
                all_img_embs, all_gal_grps = [], []
                processor = self.model.processor
                for batch in self.test_loader:
                    inputs = processor(images=batch["images"], return_tensors="pt")
                    px = inputs["pixel_values"].to(self.device, non_blocking=True)
                    with torch.amp.autocast(device_type='cuda', dtype=self.amp_dtype):
                        all_img_embs.append(self.model.encode_image(px).float().cpu())
                    all_gal_grps.extend(batch["label_groups"])
                gal_embs = torch.cat(all_img_embs, dim=0)

                q_embs = []
                for idx_q in range(0, len(golden_qs), 64):
                    inp_q = processor(
                        text=golden_qs[idx_q:idx_q + 64],
                        padding="max_length",
                        truncation=True,
                        max_length=self.model.max_text_len,
                        return_tensors="pt"
                    )
                    ids_q = inp_q["input_ids"].to(self.device, non_blocking=True)
                    mask_q = inp_q.get("attention_mask").to(self.device, non_blocking=True) if "attention_mask" in inp_q else None
                    with torch.amp.autocast(device_type='cuda', dtype=self.amp_dtype):
                        q_embs.append(self.model.encode_text(ids_q, mask_q).float().cpu())
                q_embs = torch.cat(q_embs, dim=0)

                g_metrics = compute_retrieval_metrics(q_embs, gal_embs, golden_grps, all_gal_grps)
                metrics["Golden_Recall@10"] = g_metrics["Recall@10"]
                metrics["Golden_mAP"] = g_metrics["mAP"]
                metrics["VLM_vs_Golden_R10_Gap"] = abs(metrics["Recall@10"] - g_metrics["Recall@10"])

        print(f"\n🏆 KẾT QUẢ ĐÁNH GIÁ TRÊN TẬP TEST ({len(self.test_loader.dataset):,} mẫu):")
        print(f"   • Recall@1:             {metrics['Recall@1']:.2f}%")
        print(f"   • Recall@5:             {metrics['Recall@5']:.2f}%")
        print(f"   • Recall@10 (Overall):  {metrics['Recall@10']:.2f}%")
        if "Fashion_Recall@10" in metrics:
            print(f"   • Fashion Recall@10:    {metrics['Fashion_Recall@10']:.2f}% (mAP: {metrics.get('Fashion_mAP', 0):.4f})")
        if "NonFashion_Recall@10" in metrics:
            print(f"   • NonFashion Recall@10: {metrics['NonFashion_Recall@10']:.2f}% (mAP: {metrics.get('NonFashion_mAP', 0):.4f})")
        if "Golden_Recall@10" in metrics:
            print(f"   • Golden Set Recall@10: {metrics['Golden_Recall@10']:.2f}% (Gap vs VLM: {metrics['VLM_vs_Golden_R10_Gap']:.2f}%)")
        print(f"   • mAP (Overall):        {metrics['mAP']:.4f}")
        print(f"   • Test Loss:            {metrics['loss']:.4f}")
        return metrics

In [38]:
import torch.nn as nn

def load_model_checkpoint(model: nn.Module, checkpoint_dir: str, drive_backup_dir: str = None, device: str = "cuda") -> bool:
    """
    Kiểm tra và nạp trọng số từ checkpoint cũ (ưu tiên từ Google Drive trước, sau đó tới local).
    Thứ tự ưu tiên: drive/last -> local/last -> drive/best -> local/best
    """
    import os
    import torch

    possible_paths = []
    if drive_backup_dir:
        possible_paths.append((os.path.join(drive_backup_dir, "last_model.pt"), "Last checkpoint (Google Drive)"))
    possible_paths.append((os.path.join(checkpoint_dir, "last_model.pt"), "Last checkpoint (Local SSD)"))
    if drive_backup_dir:
        possible_paths.append((os.path.join(drive_backup_dir, "best_model.pt"), "Best checkpoint (Google Drive)"))
    possible_paths.append((os.path.join(checkpoint_dir, "best_model.pt"), "Best checkpoint (Local SSD)"))

    for path, description in possible_paths:
        if os.path.exists(path):
            print(f"🔍 Tìm thấy {description} tại: {path}")
            try:
                state_dict = torch.load(path, map_location=device)
                model.load_state_dict(state_dict)
                print(f"✅ Nạp thành công trọng số từ {path} vào mô hình!")
                return True
            except Exception as e:
                print(f"⚠️ Lỗi xảy ra khi nạp checkpoint {path}: {e}")

    print("❌ Không tìm thấy hoặc không nạp được bất kỳ checkpoint 'best_model.pt' hoặc 'last_model.pt' nào.")
    return False

---
## 🚀 Phần 10: Khởi Chạy Huấn Luyện Tốc Độ Cao Trên Google Colab GPU A100 (All-In-One Execution Cell)
- **Tự động phát hiện GPU A100 (40GB / 80GB VRAM):**
  - Bật **TF32 TensorCores** + **Native BFloat16 (`torch.bfloat16`)** + **Fused AdamW**.
  - Đặt **`BATCH_SIZE = 256`** trực tiếp trên VRAM (`EVAL_BATCH_SIZE = 128`, `NUM_WORKERS = 8`).
  - Tự động tắt `gradient_checkpointing` khi VRAM $\ge 35\text{ GB}$ để tăng tốc độ mỗi epoch thêm **30% - 35%**.
- **Chống Học Vẹt Toàn Diện Trên 161,870 Ảnh:**
  - `AntiOverfitMegaClusterBatchSampler` ($k=4$ active variants/group/epoch, `sampling_weight`, $\le 2$ mẫu cùng `semantic_cluster_id`/batch).
  - `UltraECommerceLoss` Vectorized với **3-Tier False-Negative Masking** (`same_label_group | same_normalized_text | cos(txt_i, txt_j) > 0.95`).
  - Đánh giá phân tách **Fashion vs Non-Fashion** và **Golden Queries (`golden_queries_vi.jsonl`)**.

In [39]:
import os
import gc
import re
import math
import json
import time
import shutil
import random
import warnings
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from torch.utils.data import Dataset, DataLoader, Sampler
from torchvision import transforms
from collections import defaultdict, Counter
from PIL import Image
from tqdm.auto import tqdm

# ==============================================================================
# 1. KHỞI TẠO ĐƯỜNG DẪN & TỐI ƯU HÓA PHẦN CỨNG GPU A100 (AMPERE TENSORCORES)
# ==============================================================================
IS_COLAB = os.path.exists('/content')
WORK_DIR = '/content/ecommerce_cir' if IS_COLAB else '.'
DATA_DIR = f"{WORK_DIR}/data"

if IS_COLAB:
    RAW_IMG_DIR = f"{DATA_DIR}/images" if os.path.exists(f"{DATA_DIR}/images") and os.listdir(f"{DATA_DIR}/images") else f"{DATA_DIR}/raw"
else:
    RAW_IMG_DIR = "data/images" if os.path.exists("data/images") else "archive/images"

CHECKPOINT_DIR = f"{WORK_DIR}/checkpoints"
DRIVE_CANDIDATES = ['/content/drive/MyDrive/Hoc_sau_ver4', '/content/drive/MyDrive/Hoc_sau_ver_3']
DRIVE_DIR = next((p for p in DRIVE_CANDIDATES if os.path.exists(p)), '/content/drive/MyDrive/Hoc_sau_ver4')
DRIVE_CHECKPOINT_DIR = f"{DRIVE_DIR}/checkpoints" if IS_COLAB else CHECKPOINT_DIR

for p in [DATA_DIR, CHECKPOINT_DIR]:
    os.makedirs(p, exist_ok=True)

os.environ["TORCHINDUCTOR_REDUCE_OP_SPLIT"] = "0"
warnings.filterwarnings("ignore", category=UserWarning, module="torch._inductor.*")
torch.cuda.empty_cache()
gc.collect()

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
GPU_VRAM_GB = torch.cuda.get_device_properties(0).total_memory / (1024**3) if torch.cuda.is_available() else 0.0
IS_A100_OR_AMPERE = torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8
AMP_DTYPE = torch.bfloat16 if IS_A100_OR_AMPERE else torch.float16

if IS_A100_OR_AMPERE:
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True
    torch.set_float32_matmul_precision("high")

# ==============================================================================
# 2. CẤU HÌNH MODEL & SIÊU THAM SỐ TỐI ƯU (100 EPOCHS & DROPOUT)
# ==============================================================================
MODEL_NAME = "google/siglip-base-patch16-256-multilingual"

BATCH_SIZE = 256 if GPU_VRAM_GB >= 35.0 else 128
EVAL_BATCH_SIZE = 128 if GPU_VRAM_GB >= 35.0 else 64
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-2
WARMUP_STEPS = 800
MAX_EPOCHS = 100  # Nâng lên 100 epochs theo yêu cầu
DROPOUT_RATE = 0.3  # Thêm cơ chế Dropout mạnh mẽ hơn chống overfitting
NUM_WORKERS = min(8, os.cpu_count() or 4) if IS_COLAB else 2

print("=" * 80)
print(f"⚡ CẤU HÌNH PHẦN CỨNG: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'} ({GPU_VRAM_GB:.1f} GB VRAM)")
print(f"   • Chế độ tính toán: {'BFloat16 + TF32 TensorCores (Chuẩn A100)' if IS_A100_OR_AMPERE else 'Float16 Mixed Precision'}")
print(f"   • Model Backbone  : {MODEL_NAME}")
print(f"   • Batch Size      : {BATCH_SIZE} (Eval: {EVAL_BATCH_SIZE}) | Workers: {NUM_WORKERS}")
print("=" * 80)

train_file_to_use = os.path.join(DATA_DIR, "train_clean.jsonl")
val_file_to_use = os.path.join(DATA_DIR, "val_clean.jsonl")
test_file_to_use = os.path.join(DATA_DIR, "test_clean.jsonl")
golden_file_to_use = os.path.join(DATA_DIR, "golden_queries_vi.jsonl")

if not os.path.exists(train_file_to_use):
    train_file_to_use = os.path.join(DATA_DIR, "train.jsonl")
if not os.path.exists(val_file_to_use):
    val_file_to_use = os.path.join(DATA_DIR, "val.jsonl")
if not os.path.exists(test_file_to_use):
    test_file_to_use = os.path.join(DATA_DIR, "test.jsonl")

# ==============================================================================
# 3. DATASET ĐA GÓC NHÌN (MULTI-CAPTION) & MEGA-CLUSTER BATCH SAMPLER
# ==============================================================================
VIETNAMESE_SYNONYMS = {
    "áo thun": ["áo phông", "t-shirt"],
    "áo phông": ["áo thun", "t-shirt"],
    "quần jean": ["quần bò", "quần jeans"],
    "quần bò": ["quần jean", "quần jeans"],
    "quần short": ["quần đùi", "quần soóc"],
    "quần đùi": ["quần short", "quần soóc"],
    "váy": ["đầm", "váy đầm"],
    "đầm": ["váy", "váy đầm"],
    "giày thể thao": ["giày sneaker", "sneaker"],
    "sneaker": ["giày thể thao", "giày sneaker"],
    "dép": ["sandal", "dép quai ngang"],
    "sandal": ["dép quai hậu", "giày sandal"],
    "form rộng": ["dáng rộng", "oversize"],
    "dáng ôm": ["ôm body", "slimfit"]
}

IMG_RES = 256 if "256" in MODEL_NAME else 224

train_image_transform = transforms.Compose([
    transforms.RandomResizedCrop(IMG_RES, scale=(0.82, 1.0), ratio=(0.88, 1.12), interpolation=transforms.InterpolationMode.BILINEAR),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.15, hue=0.03),
    transforms.RandomRotation(degrees=8, fill=255),
    transforms.RandomPerspective(distortion_scale=0.08, p=0.25, fill=255)
])

eval_image_transform = transforms.Compose([
    transforms.Resize((IMG_RES, IMG_RES), interpolation=transforms.InterpolationMode.BILINEAR)
])

class OptimizedTextToImageDataset(Dataset):
    def __init__(self, jsonl_path: str, image_root: str, is_train: bool = True, seed: int = 42):
        self.image_root = image_root
        self.is_train = is_train
        self.samples = []
        if os.path.exists(jsonl_path):
            with open(jsonl_path, "r", encoding="utf-8") as f:
                for line in f:
                    if line.strip():
                        self.samples.append(json.loads(line))
        self.rng = random.Random(seed)
        self.synonym_patterns = [
            (re.compile(rf"\b{re.escape(term)}\b", re.IGNORECASE), syns)
            for term, syns in VIETNAMESE_SYNONYMS.items()
        ]

    def __len__(self):
        return len(self.samples)

    def _resolve_image_path(self, raw_path: str) -> str:
        if os.path.exists(raw_path):
            return raw_path
        fname = os.path.basename(raw_path)
        cand1 = os.path.join(self.image_root, fname)
        if os.path.exists(cand1):
            return cand1
        return raw_path

    def _load_image(self, img_path: str) -> Image.Image:
        full_path = self._resolve_image_path(img_path)
        try:
            with Image.open(full_path) as img:
                img_rgb = img.convert("RGB")
                return train_image_transform(img_rgb) if self.is_train else eval_image_transform(img_rgb)
        except Exception:
            return Image.new("RGB", (IMG_RES, IMG_RES), color=(240, 240, 240))

    def _augment_text_synonyms(self, text: str) -> str:
        for pat, syns in self.synonym_patterns:
            if pat.search(text) and self.rng.random() < 0.30:
                text = pat.sub(self.rng.choice(syns), text, count=1)
        return text

    def _build_rich_text(self, item: dict) -> str:
        title = item.get("title", "").strip()
        captions = item.get("captions") or {}
        category = item.get("category", "").strip()
        attrs = item.get("attributes") or {}
        attr_parts = [f"{v}" for k, v in attrs.items() if v and k in ["color", "pattern", "material", "style", "article_type"]]
        attr_str = ", ".join(attr_parts)

        if not self.is_train:
            eval_base = captions.get("query") or title
            parts = []
            if category:
                parts.append(f"[{category}]")
            parts.append(eval_base)
            if attr_str and attr_str.lower() not in eval_base.lower():
                parts.append(f"({attr_str})")
            return " ".join(parts)[:256]

        candidates = []
        if title:
            candidates.append(title)
            if category:
                candidates.append(f"[{category}] {title}")
            if attr_str and attr_str.lower() not in title.lower():
                candidates.append(f"{title} ({attr_str})")

        if isinstance(captions, dict):
            for key in ("query", "detail", "attrs", "distinguishing"):
                val = captions.get(key)
                if val and isinstance(val, str) and len(val.strip()) >= 6:
                    v_clean = val.strip()
                    candidates.append(v_clean)
                    if category and key in ("query", "detail"):
                        candidates.append(f"[{category}] {v_clean}")

        chosen = self.rng.choice(candidates) if candidates else title
        return self._augment_text_synonyms(chosen)[:256]

    def __getitem__(self, idx: int):
        item = self.samples[idx]
        pid = str(item.get("product_id", idx))
        src = item.get("source") or (pid.split("_")[0] if "_" in pid else "unknown")
        cat = item.get("category", "unknown")
        return {
            "image": self._load_image(item.get("image_path", "")),
            "text": self._build_rich_text(item),
            "label_group": item.get("label_group", f"group_{idx}"),
            "product_id": pid,
            "category": cat,
            "source": src,
            "semantic_cluster_id": item.get("semantic_cluster_id", f"{src}::{cat}")
        }

def ecom_collate_fn(batch):
    return {
        "images": [b["image"] for b in batch],
        "texts": [b["text"] for b in batch],
        "label_groups": [b["label_group"] for b in batch],
        "product_ids": [b["product_id"] for b in batch],
        "categories": [b["category"] for b in batch],
        "sources": [b.get("source", "unknown") for b in batch],
        "semantic_cluster_ids": [b.get("semantic_cluster_id", "default") for b in batch]
    }

if "AntiOverfitMegaClusterBatchSampler" not in globals():
    class AntiOverfitMegaClusterBatchSampler(Sampler):
        def __init__(self, samples: list, batch_size: int = 256, max_active_per_group: int = 4, max_per_cluster_in_batch: int = 2, seed: int = 42):
            self.samples = samples
            self.batch_size = batch_size
            self.max_active_per_group = max_active_per_group
            self.max_per_cluster_in_batch = max_per_cluster_in_batch
            self.seed = seed
            self.epoch = 0
            self.group_to_indices = defaultdict(list)
            self.index_to_cluster = {}
            self.index_to_weight = np.ones(len(samples), dtype=np.float64)
            for idx, item in enumerate(samples):
                grp = item.get("label_group") or str(item.get("product_id", idx))
                clust = item.get("semantic_cluster_id") or item.get("category", "unknown")
                self.group_to_indices[grp].append(idx)
                self.index_to_cluster[idx] = clust
                self.index_to_weight[idx] = float(item.get("sampling_weight", 1.0))
            est_active = sum(min(len(idxs), self.max_active_per_group) for idxs in self.group_to_indices.values())
            self.num_batches = max(1, est_active // self.batch_size)

        def set_epoch(self, epoch: int):
            self.epoch = epoch

        def __len__(self):
            return self.num_batches

        def __iter__(self):
            rng = np.random.default_rng(self.seed + self.epoch)
            active_indices = []
            for grp, idxs in self.group_to_indices.items():
                k = self.max_active_per_group
                if len(idxs) <= k:
                    active_indices.extend(idxs)
                else:
                    start = (self.epoch * k) % len(idxs)
                    active_indices.extend([idxs[(start + offset) % len(idxs)] for offset in range(k)])
            active_indices = np.array(active_indices, dtype=np.int64)
            weights = self.index_to_weight[active_indices]
            weights = weights / weights.sum()
            ordered = rng.choice(active_indices, size=len(active_indices), replace=False, p=weights).tolist()
            batches, deferred, current_batch = [], [], []
            cluster_counts = defaultdict(int)
            for idx in ordered:
                c_id = self.index_to_cluster[idx]
                if cluster_counts[c_id] < self.max_per_cluster_in_batch:
                    current_batch.append(idx)
                    cluster_counts[c_id] += 1
                    if len(current_batch) == self.batch_size:
                        batches.append(current_batch)
                        current_batch = []
                        cluster_counts.clear()
                else:
                    deferred.append(idx)
            for idx in deferred:
                current_batch.append(idx)
                if len(current_batch) == self.batch_size:
                    batches.append(current_batch)
                    current_batch = []
            return iter(batches[:self.num_batches])

train_dataset = OptimizedTextToImageDataset(jsonl_path=train_file_to_use, image_root=RAW_IMG_DIR, is_train=True, seed=42)
val_dataset = OptimizedTextToImageDataset(jsonl_path=val_file_to_use, image_root=RAW_IMG_DIR, is_train=False, seed=42)
test_dataset = OptimizedTextToImageDataset(jsonl_path=test_file_to_use, image_root=RAW_IMG_DIR, is_train=False, seed=42)

train_batch_sampler = AntiOverfitMegaClusterBatchSampler(
    samples=train_dataset.samples,
    batch_size=BATCH_SIZE,
    max_active_per_group=4,
    max_per_cluster_in_batch=2,
    seed=42
)

total_loaded = max(1, len(train_dataset) + len(val_dataset) + len(test_dataset))
print("=" * 80)
print("📊 CẤU HÌNH DỮ LIỆU TOÀN VẸN 161,870 ẢNH (ANTI-OVERFITTING & MEGA-CLUSTER):")
print(f"   • Train Dataset : {len(train_dataset):>7,} mẫu ({len(train_dataset)/total_loaded*100:.2f}%) | Active Batches/Epoch: {len(train_batch_sampler):,}")
print(f"   • Val Dataset   : {len(val_dataset):>7,} mẫu ({len(val_dataset)/total_loaded*100:.2f}%) -> {os.path.basename(val_file_to_use)}")
print(f"   • Test Dataset  : {len(test_dataset):>7,} mẫu ({len(test_dataset)/total_loaded*100:.2f}%) -> {os.path.basename(test_file_to_use)}")
print("=" * 80)


⚡ CẤU HÌNH PHẦN CỨNG: NVIDIA A100-SXM4-40GB (39.5 GB VRAM)
   • Chế độ tính toán: BFloat16 + TF32 TensorCores (Chuẩn A100)
   • Model Backbone  : google/siglip-base-patch16-256-multilingual
   • Batch Size      : 256 (Eval: 128) | Workers: 8
📊 CẤU HÌNH DỮ LIỆU TOÀN VẸN 161,870 ẢNH (ANTI-OVERFITTING & MEGA-CLUSTER):
   • Train Dataset : 153,126 mẫu (94.60%) | Active Batches/Epoch: 553
   • Val Dataset   :   4,372 mẫu (2.70%) -> val_clean.jsonl
   • Test Dataset  :   4,372 mẫu (2.70%) -> test_clean.jsonl


In [42]:
import os
import gc
import json
import torch
from torch.utils.data import DataLoader

# ==============================================================================
# 1. KHỞI TẠO DATALOADERS TỐI ƯU HÓA HIGH-THROUGHPUT
# ==============================================================================
loader_kwargs = {
    "num_workers": NUM_WORKERS,
    "pin_memory": True,
    "collate_fn": ecom_collate_fn,
    "persistent_workers": True if NUM_WORKERS > 0 else False,
    "prefetch_factor": 4 if NUM_WORKERS > 0 else None
}

train_loader = DataLoader(
    train_dataset,
    batch_sampler=train_batch_sampler,
    **loader_kwargs
)

val_loader = DataLoader(
    val_dataset,
    batch_size=EVAL_BATCH_SIZE,
    shuffle=False,
    **loader_kwargs
)

test_loader = DataLoader(
    test_dataset,
    batch_size=EVAL_BATCH_SIZE,
    shuffle=False,
    **loader_kwargs
)

# ==============================================================================
# 2. KHỞI TẠO SIÊU MÔ HÌNH VÀ HÀM PHẠT CHỐNG OVERFITTING
# ==============================================================================
print("-> Đang khởi tạo mô hình và tải cấu hình tối ưu...")
model = HighlyEffectiveSigLIP(
    model_name=MODEL_NAME,
    unfreeze_last_blocks=8,  # Mở khóa sâu 8 blocks để tối đa hóa mAP
    max_length=128,
    device=DEVICE
).to(DEVICE)

# Sửa lỗi kích thước vị trí (size mismatch) để tương thích với checkpoint max_length=128
expand_siglip_position_embeddings(model, target_max_length=128)

loss_fn = EnhancedECommerceLoss(
    init_temp=10.0,
    init_bias=-10.0,
    triplet_margin=0.25,
    weight_triplet=0.5,
    dropout_rate=DROPOUT_RATE,
    semantic_mask_thresh=0.95
).to(DEVICE)

# ==============================================================================
# 3. KHỞI TẠO TRAINER & TỰ ĐỘNG KHÔI PHỤC (RESUME CHECKPOINT)
# ==============================================================================
trainer = MultimodalTrainer(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    test_loader=test_loader,
    loss_fn=loss_fn,
    device=DEVICE,
    batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    max_epochs=MAX_EPOCHS,
    warmup_steps=WARMUP_STEPS,
    checkpoint_dir=CHECKPOINT_DIR,
    drive_backup_dir=DRIVE_CHECKPOINT_DIR,
    patience=10  # Tăng patience từ 5 lên 10 theo yêu cầu
)

# Tự động nạp lại checkpoint nếu bị ngắt kết nối đột ngột trên Colab
has_checkpoint = load_model_checkpoint(model, CHECKPOINT_DIR, DRIVE_CHECKPOINT_DIR, device=DEVICE)
start_epoch = 1

if has_checkpoint:
    history_path = os.path.join(CHECKPOINT_DIR, "train_history.json")
    if os.path.exists(history_path):
        try:
            with open(history_path, "r", encoding="utf-8") as f:
                history_data = json.load(f)
            if "epoch" in history_data and len(history_data["epoch"]) > 0:
                start_epoch = max(history_data["epoch"]) + 1
                trainer.train_history = history_data
                print(f"-> Đã nạp lịch sử huấn luyện cũ. Sẽ tiếp tục học từ Epoch {start_epoch}.")
        except Exception as e:
            print(f"⚠️ Không thể nạp lịch sử huấn luyện cũ: {e}")

torch.cuda.empty_cache()
gc.collect()
print("✅ Khởi tạo bộ Trainer thành công! Chạy ô tiếp theo để bắt đầu huấn luyện.")

-> Đang khởi tạo mô hình và tải cấu hình tối ưu...
[Upgrade] Đang khởi tạo mô hình từ: google/siglip-base-patch16-256-multilingual...


Loading weights:   0%|          | 0/408 [00:00<?, ?it/s]

⚡ Đã kích hoạt Gradient Checkpointing để tiết kiệm VRAM.
📊 Tham số huấn luyện được: 122,069,249 (32.18%)
⚠️ Đang mở rộng Text Position Embeddings từ 64 lên 128 tokens...
✅ Đã mở rộng thành công lên 128! Sẵn sàng huấn luyện với max_length = 128.
🔍 Tìm thấy Last checkpoint (Google Drive) tại: /content/drive/MyDrive/Hoc_sau_ver4/checkpoints/last_model.pt
✅ Nạp thành công trọng số từ /content/drive/MyDrive/Hoc_sau_ver4/checkpoints/last_model.pt vào mô hình!
-> Đã nạp lịch sử huấn luyện cũ. Sẽ tiếp tục học từ Epoch 25.
✅ Khởi tạo bộ Trainer thành công! Chạy ô tiếp theo để bắt đầu huấn luyện.


In [ ]:
import time
import torch
import gc
import os
import shutil

# ==============================================================================
# ĐẮP VÁ KHẨN CẤP (MONKEY PATCH) CHO HIGHLYEFFECTIVESIGLIP NẾU THIẾU HÀM FORWARD
# ==============================================================================
if 'model' in globals():
    cls = model.__class__
    if not hasattr(cls, 'forward') or cls.forward == torch.nn.Module.forward:
        print("⚙️ Phát hiện HighlyEffectiveSigLIP thiếu hàm forward! Đang vá nóng các phương thức...")

        def encode_image(self, pixel_values):
            vision_outputs = self.siglip.vision_model(pixel_values=pixel_values)
            img_feats = vision_outputs[1] if len(vision_outputs) > 1 else vision_outputs.pooler_output
            return torch.nn.functional.normalize(img_feats, p=2, dim=-1)

        def encode_text(self, input_ids, attention_mask=None):
            text_outputs = self.siglip.text_model(input_ids=input_ids, attention_mask=attention_mask)
            txt_feats = text_outputs[1] if len(text_outputs) > 1 else text_outputs.pooler_output
            return torch.nn.functional.normalize(txt_feats, p=2, dim=-1)

        def forward(self, pixel_values, input_ids, attention_mask=None):
            img_emb = self.encode_image(pixel_values)
            txt_emb = self.encode_text(input_ids, attention_mask)
            return img_emb, txt_emb

        def compose_query(self, ref_pixels, mod_ids, mod_mask=None):
            ref_img_emb = self.encode_image(ref_pixels)
            mod_txt_emb = self.encode_text(mod_ids, mod_mask)
            return self.cir_combiner(ref_img_emb, mod_txt_emb)

        cls.encode_image = encode_image
        cls.encode_text = encode_text
        cls.forward = forward
        cls.compose_query = compose_query
        print("✅ Vá nóng HighlyEffectiveSigLIP thành công!")

# Đảm bảo các biến cấu hình luôn tồn tại kể cả khi chạy độc lập
if 'start_epoch' not in globals():
    start_epoch = 1
if 'MAX_EPOCHS' not in globals():
    MAX_EPOCHS = 100

# Khởi tạo khẩn cấp bộ trainer nếu chưa được định nghĩa trước đó
if 'trainer' not in globals():
    print("⚠️ Không tìm thấy biến 'trainer' trong phiên làm việc. Tiến hành khởi tạo bộ huấn luyện...")
    if 'model' not in globals():
        raise NameError("Không tìm thấy biến 'model'. Vui lòng chạy lại các ô cài đặt mô hình trước.")
    if 'train_loader' not in globals() or 'val_loader' not in globals():
        raise NameError("Không tìm thấy DataLoaders. Vui lòng chạy lại các ô nạp dữ liệu trước.")
    if 'loss_fn' not in globals():
        loss_fn = EnhancedECommerceLoss(
            init_temp=10.0,
            init_bias=-10.0,
            triplet_margin=0.25,
            weight_triplet=0.5,
            dropout_rate=0.3,
            semantic_mask_thresh=0.95
        ).to(DEVICE)

    trainer = MultimodalTrainer(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        test_loader=test_loader if 'test_loader' in globals() else None,
        loss_fn=loss_fn,
        device=DEVICE if 'DEVICE' in globals() else 'cuda',
        batch_size=BATCH_SIZE if 'BATCH_SIZE' in globals() else 128,
        learning_rate=LEARNING_RATE if 'LEARNING_RATE' in globals() else 1e-4,
        weight_decay=WEIGHT_DECAY if 'WEIGHT_DECAY' in globals() else 1e-2,
        max_epochs=MAX_EPOCHS,
        warmup_steps=WARMUP_STEPS if 'WARMUP_STEPS' in globals() else 800,
        checkpoint_dir=CHECKPOINT_DIR if 'CHECKPOINT_DIR' in globals() else 'checkpoints',
        drive_backup_dir=DRIVE_CHECKPOINT_DIR if 'DRIVE_CHECKPOINT_DIR' in globals() else None,
        patience=10  # Tăng patience từ 5 lên 10 theo yêu cầu
    )
    print("✅ Khởi tạo bộ Trainer thành công!")

print(f"🚀 BẮT ĐẦU VÒNG LẶP HUẤN LUYỆN CHÍNH THỨC TỪ EPOCH {start_epoch} ĐẾN {MAX_EPOCHS}...")

for epoch in range(start_epoch, MAX_EPOCHS + 1):
    start_time = time.time()

    # 1. Huấn luyện 1 Epoch
    train_loss = trainer.train_epoch(epoch)

    # 2. Đánh giá trên tập Validation
    val_metrics = trainer.evaluate(val_loader, desc=f"Epoch {epoch} [Val]")

    epoch_time = time.time() - start_time

    # 3. Ghi nhận lịch sử huấn luyện
    trainer.train_history["epoch"].append(epoch)
    trainer.train_history["train_loss"].append(train_loss)
    trainer.train_history["val_loss"].append(val_metrics["loss"])
    trainer.train_history["val_r1"].append(val_metrics["Recall@1"])
    trainer.train_history["val_r5"].append(val_metrics["Recall@5"])
    trainer.train_history["val_r10"].append(val_metrics["Recall@10"])
    trainer.train_history["val_map"].append(val_metrics["mAP"])
    trainer.train_history["epoch_time"].append(epoch_time)

    # 4. Lưu lại lịch sử huấn luyện dạng JSON
    trainer.save_history()

    # 5. Lưu checkpoint của Epoch cuối cùng (Last Checkpoint)
    torch.save(model.state_dict(), trainer.last_checkpoint_path)
    if trainer.drive_backup_dir and os.path.exists(trainer.drive_backup_dir):
        shutil.copy2(trainer.last_checkpoint_path, os.path.join(trainer.drive_backup_dir, "last_model.pt"))

    # In báo cáo kết quả Epoch
    print(f"\n⭐ [Epoch {epoch}/{MAX_EPOCHS}] Hoàn thành trong {epoch_time:.1f}s:")
    print(f"   • Train Loss:     {train_loss:.4f}")
    print(f"   • Val Loss:       {val_metrics['loss']:.4f}")
    print(f"   • Val Recall@10:  {val_metrics['Recall@10']:.2f}%")
    print(f"   • Val mAP:        {val_metrics['mAP']:.4f}")
    if "Fashion_Recall@10" in val_metrics:
        print(f"   • Fashion R@10:   {val_metrics['Fashion_Recall@10']:.2f}% | Non-Fashion R@10: {val_metrics.get('NonFashion_Recall@10', 0.0):.2f}%")

    # 6. Kiểm tra cải thiện & Early Stopping dựa trên Val Loss
    is_best = trainer.early_stopping.step(val_metrics["loss"], epoch, model, backup_dir=trainer.drive_backup_dir)
    if is_best:
         print(f"   ✨ Phát hiện mô hình tốt nhất mới tại Epoch {epoch}! Đã đồng bộ checkpoint lên Google Drive.")

    if trainer.early_stopping.early_stop:
         break

    # Giải phóng bộ nhớ cache GPU
    torch.cuda.empty_cache()
    gc.collect()
    print("-" * 80)

print("\n🎉 QUÁ TRÌNH HUẤN LUYỆN ĐÃ KẾT THÚC HOẶC ĐÃ ĐẠT ĐIỂM DỪNG SỚM!")

🚀 BẮT ĐẦU VÒNG LẶP HUẤN LUYỆN CHÍNH THỨC TỪ EPOCH 25 ĐẾN 100...


Epoch 25/100 [Train BF16-A100]:   0%|          | 0/553 [00:01<?, ?it/s]

### 🛠️ Giải Pháp: Mở Rộng Nhúng Vị Trí (Position Embeddings) để Giữ `max_length = 128`

Mô hình `google/siglip-base-patch16-256-multilingual` mặc định chỉ hỗ trợ `max_position_embeddings = 64`. Khi ta token hóa đầu vào thành 128 tokens, mô hình sẽ ném lỗi `ValueError` vì không có trọng số vị trí từ 64 đến 127.

Để khắc phục, chúng ta cần can thiệp trực tiếp vào lớp nhúng văn bản (`text_model.embeddings`):
1. Tạo một ma trận nhúng vị trí mới có kích thước `(128, embed_dim)` thay thế cho ma trận cũ `(64, embed_dim)`.
2. Sao chép các trọng số vị trí đã học trước đó từ vị trí 0-63 sang ma trận mới để không làm mất thông tin cũ.
3. Khởi tạo các vị trí mới từ 64-127 bằng cách lặp lại (hoặc nội suy) trọng số của vị trí cuối cùng.
4. Cập nhật `position_ids` và cấu hình hệ thống `max_position_embeddings = 128`.

---
## 📊 Phần 11: Trực Quan Hóa Quá Trình Huấn Luyện (Loss & Metrics Curves)
Vẽ đồ thị khoa học theo dõi sự hội tụ của mô hình:
1. `Train Loss` vs `Val Loss` qua các epoch.
2. `Recall@1`, `Recall@5`, `Recall@10` và `mAP` trên tập Validation.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

epochs = train_history["epoch"]

# Đồ thị 1: Train Loss vs Val Loss
axes[0].plot(epochs, train_history["train_loss"], 'o-', color='#1f77b4', linewidth=2.5, label='Train Loss')
axes[0].plot(epochs, train_history["val_loss"], 's--', color='#d62728', linewidth=2.5, label='Val Loss')
axes[0].set_title('Đường Cong Hàm Mất Mát (Loss Curves)', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('Loss', fontsize=12)
axes[0].legend(fontsize=12)
axes[0].grid(True, linestyle='--', alpha=0.7)

# Đồ thị 2: Validation Retrieval Metrics (Recall@K & mAP)
axes[1].plot(epochs, train_history["val_r1"], 'o-', color='#2ca02c', linewidth=2, label='Recall@1 (%)')
axes[1].plot(epochs, train_history["val_r5"], '^-', color='#ff7f0e', linewidth=2, label='Recall@5 (%)')
axes[1].plot(epochs, train_history["val_r10"], 'd-', color='#9467bd', linewidth=2.5, label='Recall@10 (%)')
axes[1].plot(epochs, [m * 100 for m in train_history["val_map"]], 'x--', color='#8c564b', linewidth=2, label='mAP x 100')
axes[1].set_title('Chỉ Số Đánh Giá Tra Cứu (Validation Metrics)', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_ylabel('Điểm số (%)', fontsize=12)
axes[1].legend(fontsize=11)
axes[1].grid(True, linestyle='--', alpha=0.7)

plt.tight_layout()
chart_path = f"{OUTPUT_DIR}/training_metrics.png"
plt.savefig(chart_path, dpi=300)
plt.show()
print(f"✅ Đã lưu biểu đồ vào: {chart_path}")

### 🛠️ Giả Lập Vector Database (Fake FAISS Index & Metadata) để Kiểm Thử Nhanh
Đoạn code dưới đây sẽ tạo ra một index FAISS giả lập chứa các vector ngẫu nhiên (synthetic embeddings) cùng với danh sách thông tin sản phẩm mẫu. Điều này cho phép bạn chạy thử nghiệm tính năng tìm kiếm Text-to-Image và CIR ngay lập tức mà không cần trích xuất cơ sở dữ liệu thật.

In [ ]:
import numpy as np
import faiss
import json
import time

# 1. Cấu hình thông số giả lập
fake_num_products = 500  # Số lượng sản phẩm giả lập
vector_dim = getattr(model, 'embed_dim', 768)  # Tự động khớp chiều vector của Model (768 hoặc 1152)

print(f"-> Đang tạo {fake_num_products} vector ngẫu nhiên chuẩn hóa L2...")
# Tạo ma trận vector ngẫu nhiên
rng = np.random.default_rng(seed=42)
fake_embeddings = rng.standard_normal((fake_num_products, vector_dim)).astype(np.float32)
faiss.normalize_L2(fake_embeddings)

# 2. Xây dựng danh sách thông tin sản phẩm mẫu (Fake Metadata)
fake_titles = [
    "Áo sơ mi trắng nam công sở dáng ôm slim fit cotton chống nhăn",
    "Quần jean bò nữ màu xanh dương cạp cao tôn dáng ống suông rộng",
    "Áo thun phông nam cổ tròn màu đen cá tính vải thun co giãn 4 chiều",
    "Đầm váy nữ dáng xòe màu đỏ dự tiệc chất liệu lụa cao cấp",
    "Bộ đồ thể thao nam mùa hè năng động thoáng khí",
    "Giày sneaker thể thao nam nữ màu trắng phong cách Hàn Quốc",
    "Túi xách nữ thời trang màu nâu pastel sang trọng quý phái",
    "Kính râm chống tia UV phong cách unisex thời thượng"
]

fake_categories = ["áo sơ mi", "quần jean", "áo thun", "váy đầm", "đồ thể thao", "giày dép", "túi xách", "phụ kiện"]

fake_metadata = []
for i in range(fake_num_products):
    # Lựa chọn ngẫu nhiên tiêu đề và danh mục mẫu
    title_template = fake_titles[i % len(fake_titles)]
    category = fake_categories[i % len(fake_categories)]

    fake_metadata.append({
        "product_id": f"fake_prod_{10000 + i}",
        "title": f"[Fake Brand] {title_template} - Phiên bản v{i}",
        "category": f"fake_group_{i % 10}",
        "price": int(rng.integers(150000, 850000))
    })

# 3. Tạo Index FAISS FlatL2 (hoặc HNSW tùy chọn)
print("-> Khởi tạo FAISS Index FlatL2 cho dữ liệu giả lập...")
fake_faiss_index = faiss.IndexFlatIP(vector_dim) # Inner Product tương đương Cosine Similarity trên L2
fake_faiss_index.add(fake_embeddings)

# 4. Hàm tìm kiếm thử nghiệm trên dữ liệu giả
def search_fake_catalog(query_text: str, top_k: int = 5):
    model.eval()
    import torch
    with torch.no_grad():
        # Trích xuất vector truy vấn thực tế từ mô hình
        inp = model.processor(
            text=[query_text],
            padding="max_length",
            truncation=True,
            max_length=128,
            return_tensors="pt"
        )
        txt_emb = model.encode_text(inp["input_ids"].to(DEVICE), inp.get("attention_mask")).cpu().numpy()

        # Tìm kiếm trên index giả lập
        scores, indices = fake_faiss_index.search(txt_emb, top_k)

    print("=" * 80)
    print(f"🔍 KẾT QUẢ TÌM KIẾM GIẢ LẬP CHO TRUY VẤN: '{query_text}'")
    print("=" * 80)
    for rank, (score, idx) in enumerate(zip(scores[0], indices[0]), start=1):
        item = fake_metadata[idx]
        print(f"[{rank:02d}] Độ tương đồng: {score:.4f} | Mã SP: {item.get('product_id')}")
        print(f"     Tiêu đề: {item.get('title')}")
        print(f"     Giá: {item.get('price'):,} VNĐ")
        print("-" * 80)

# 5. Chạy thử nghiệm ngay tại chỗ
search_fake_catalog("áo sơ mi trắng nam công sở dáng slim fit", top_k=3)

---
## 🔍 Phần 12: Thử Nghiệm Tra Cứu Trực Tiếp (Sanity Inference: CIR & Text Search)
Trực quan hóa kết quả tìm kiếm thực tế:
- **Composed Image Retrieval (CIR)**: Ảnh tham chiếu $I_{ref}$ + Câu yêu cầu chỉnh sửa $T_{mod} \rightarrow$ Dự đoán vector embedding mục tiêu $z_{query}$.

In [ ]:
@torch.no_grad()
def demo_composed_retrieval(model, val_dataset, query_idx: int = 0, modifier_query: str = "đổi sang màu đỏ"):
    """Thử nghiệm trực quan Composed Image Retrieval (CIR)."""
    model.eval()
    ref_item = val_dataset.samples[query_idx]
    ref_img = val_dataset._load_image(ref_item.get("image_path", ""))

    processor = model.processor
    inputs_img = processor(images=[ref_img], return_tensors="pt")
    max_len = getattr(model, "max_text_len", 16)
    inputs_txt = processor(text=[modifier_query], padding="max_length", truncation=True, max_length=max_len, return_tensors="pt")

    pixel_values = inputs_img["pixel_values"].to(model.device)
    input_ids = inputs_txt["input_ids"].to(model.device)

    # Sinh Composed Query Embedding: z_query = f(z_img, z_text_modifier)
    query_emb = model.compose_query(pixel_values, input_ids)

    print("=" * 65)
    print("🔎 DEMO COMPOSED IMAGE RETRIEVAL (CIR):")
    print(f"   • Ảnh tham chiếu:        {ref_item.get('title')}")
    print(f"   • Yêu cầu chỉnh sửa:    '{modifier_query}'")
    print(f"   • Composed Vector shape: {query_emb.shape} (Chuẩn hóa L2 thành công!)")
    print("=" * 65)

demo_composed_retrieval(model, val_dataset, query_idx=0, modifier_query="kiểu dáng này nhưng đổi sang màu đen thanh lịch")

---
## ⚡ Phần 13: Xây Dựng Vector Database FAISS HNSW (Hierarchical Navigable Small World)
- Sử dụng mô hình SigLIP đã fine-tune để trích xuất toàn bộ vector nhúng (embedding) của kho hàng (Gallery / Test Dataset).
- Khởi tạo chỉ mục **FAISS IndexHNSWFlat** với độ đo `METRIC_INNER_PRODUCT` (tương đương Cosine Similarity trên vector chuẩn hóa $L_2$).
- Thiết lập siêu tham số đồ thị HNSW:
  - $M = 32$ (Số liên kết láng giềng trên mỗi đỉnh đồ thị).
  - $efConstruction = 128$ (Độ sâu tìm kiếm trong quá trình dựng đồ thị index).
  - $efSearch = 64$ (Độ sâu tìm kiếm khi truy vấn thời gian thực).
- Lưu trữ chỉ mục `faiss_hnsw_index.bin` và metadata sản phẩm `product_metadata.json` ra Google Drive.

In [ ]:
import faiss

print("-> Bắt đầu trích xuất toàn bộ Embeddings của tập Test/Gallery bằng SigLIP...")
model.eval()
gallery_embeddings = []
gallery_metadata = []
processor = model.processor

with torch.no_grad():
    for batch in tqdm(test_loader, desc="Extracting Gallery Embeddings"):
        # Sử dụng processor của mô hình để chuyển danh sách ảnh PIL thành tensor pixel_values chuẩn hóa
        inputs = processor(images=batch["images"], return_tensors="pt")
        pixel_values = inputs["pixel_values"].to(DEVICE)

        with torch.amp.autocast(device_type="cuda", dtype=AMP_DTYPE):
            img_feats = model.encode_image(pixel_values)

        gallery_embeddings.append(img_feats.cpu().numpy())
        # Lưu lại siêu dữ liệu sản phẩm tương ứng
        for i in range(len(batch["product_ids"])):
            gallery_metadata.append({
                "product_id": batch["product_ids"][i],
                "title": batch["texts"][i],
                "category": batch["label_groups"][i]
            })

gallery_embeddings = np.concatenate(gallery_embeddings, axis=0).astype(np.float32)
faiss.normalize_L2(gallery_embeddings)

dim = gallery_embeddings.shape[1]
print(f"✅ Đã trích xuất {len(gallery_embeddings):,} vectors (Dimension = {dim})")

# Khởi tạo FAISS HNSW Index
M = 32
efConstruction = 128
efSearch = 64

faiss_index = faiss.IndexHNSWFlat(dim, M, faiss.METRIC_INNER_PRODUCT)
faiss_index.hnsw.efConstruction = efConstruction
faiss_index.hnsw.efSearch = efSearch

t_start = time.perf_counter()
faiss_index.add(gallery_embeddings)
t_end = time.perf_counter()
print(f"✅ Đã xây dựng thành công FAISS HNSW Index với {faiss_index.ntotal:,} sản phẩm trong {(t_end - t_start):.3f}s!")

# Lưu Index và Metadata ra ổ đĩa
index_save_path = f"{OUTPUT_DIR}/faiss_hnsw_index.bin"
metadata_save_path = f"{OUTPUT_DIR}/product_metadata.json"
faiss.write_index(faiss_index, index_save_path)
with open(metadata_save_path, "w", encoding="utf-8") as f:
    json.dump(gallery_metadata, f, ensure_ascii=False, indent=2)

print(f"✅ Đã lưu FAISS Index vào: {index_save_path}")
print(f"✅ Đã lưu Metadata vào:     {metadata_save_path}")

# Đồng bộ sao lưu lên Google Drive
if os.path.exists(DRIVE_CHECKPOINT_DIR):
    shutil.copy2(index_save_path, os.path.join(DRIVE_CHECKPOINT_DIR, "faiss_hnsw_index.bin"))
    shutil.copy2(metadata_save_path, os.path.join(DRIVE_CHECKPOINT_DIR, "product_metadata.json"))
    print("✅ Đã đồng bộ FAISS Index và Metadata lên Google Drive thành công!")

---
## ⏱️ Phần 14: Đo Đạc Độ Trễ Toàn Trình (Latency Benchmark - Đảm Bảo < 200 ms)
Kiểm định nghiêm ngặt yêu cầu kỹ thuật trong `context.md` (**Latency < 200 ms / query**):
1. **Search Latency:** Đo tốc độ truy vấn thuần trên đồ thị FAISS HNSW ($Q = 1,000$ truy vấn).
2. **Text-to-Image Latency:** Tokenize + SigLIP Text Tower + FAISS Search.
3. **CIR Latency:** Image Tower + Text Modifier + CIR Combiner Module + FAISS Search.

In [ ]:
# 1. Đo đạc Search Latency thuần của FAISS HNSW Index (Top-10)
NUM_TEST_QUERIES = 1000
TOP_K = 10

rng = np.random.default_rng(seed=42)
test_query_vectors = rng.standard_normal((NUM_TEST_QUERIES + 50, dim)).astype(np.float32)
faiss.normalize_L2(test_query_vectors)

# Warmup cache
for i in range(50):
    faiss_index.search(test_query_vectors[i:i+1], TOP_K)

# Benchmark 1000 queries
faiss_latencies = []
for i in range(50, 50 + NUM_TEST_QUERIES):
    q = test_query_vectors[i:i+1]
    t0 = time.perf_counter()
    faiss_index.search(q, TOP_K)
    t1 = time.perf_counter()
    faiss_latencies.append((t1 - t0) * 1000.0)

faiss_latencies = np.array(faiss_latencies)

# 2. Đo đạc Text-to-Image Query Latency (End-to-End)
sample_queries = [
    "áo khoác dạ dáng dài nam màu đen thanh lịch",
    "quần jean bò nữ màu xanh dương cạp cao tôn dáng",
    "đầm hoa nữ mùa hè dáng xòe vải voan mềm mại",
    "áo sơ mi nam công sở trắng cổ cài cúc slim fit",
    "bộ đồ bơi nữ đi biển họa tiết hoa sặc sỡ"
]

text_e2e_latencies = []
model.eval()
with torch.no_grad():
    # Warmup
    inp = model.processor(text=[sample_queries[0]], padding="max_length", truncation=True, max_length=getattr(model, "max_text_len", 16), return_tensors="pt")
    _ = model.encode_text(inp["input_ids"].to(DEVICE), inp.get("attention_mask"))

    for _ in range(100):
        query_text = random.choice(sample_queries)
        t0 = time.perf_counter()
        inp = model.processor(text=[query_text], padding="max_length", truncation=True, max_length=getattr(model, "max_text_len", 16), return_tensors="pt")
        txt_emb = model.encode_text(inp["input_ids"].to(DEVICE), inp.get("attention_mask")).cpu().numpy()
        scores, indices = faiss_index.search(txt_emb, TOP_K)
        t1 = time.perf_counter()
        text_e2e_latencies.append((t1 - t0) * 1000.0)

text_e2e_latencies = np.array(text_e2e_latencies)

# 3. Đo đạc CIR Latency (End-to-End)
cir_e2e_latencies = []
dummy_ref_img = test_dataset._load_image(test_dataset.samples[0].get("image_path", ""))
with torch.no_grad():
    inp_img = model.processor(images=[dummy_ref_img], return_tensors="pt")
    inp_mod = model.processor(text=["đổi sang màu đỏ"], padding="max_length", truncation=True, max_length=getattr(model, "max_text_len", 16), return_tensors="pt")
    px = inp_img["pixel_values"].to(DEVICE)
    tx = inp_mod["input_ids"].to(DEVICE)

    # Warmup
    _ = model.compose_query(px, tx)

    for _ in range(100):
        t0 = time.perf_counter()
        q_emb = model.compose_query(px, tx).cpu().numpy()
        scores, indices = faiss_index.search(q_emb, TOP_K)
        t1 = time.perf_counter()
        cir_e2e_latencies.append((t1 - t0) * 1000.0)

cir_e2e_latencies = np.array(cir_e2e_latencies)

print("=" * 80)
print("📊 BẢNG TỔNG HỢP ĐO ĐẠC ĐỘ TRỄ HỆ THỐNG (LATENCY BENCHMARK):")
print("=" * 80)
print(f"{'Quy trình (Pipeline)':<35} | {'Mean (ms)':<10} | {'P50 (ms)':<10} | {'P95 (ms)':<10} | {'P99 (ms)':<10}")
print("-" * 80)
print(f"{'FAISS HNSW Search thuần (Top-10)':<35} | {np.mean(faiss_latencies):<10.3f} | {np.percentile(faiss_latencies, 50):<10.3f} | {np.percentile(faiss_latencies, 95):<10.3f} | {np.percentile(faiss_latencies, 99):<10.3f}")
print(f"{'Text-to-Image E2E Latency':<35} | {np.mean(text_e2e_latencies):<10.3f} | {np.percentile(text_e2e_latencies, 50):<10.3f} | {np.percentile(text_e2e_latencies, 95):<10.3f} | {np.percentile(text_e2e_latencies, 99):<10.3f}")
print(f"{'CIR Query E2E Latency':<35} | {np.mean(cir_e2e_latencies):<10.3f} | {np.percentile(cir_e2e_latencies, 50):<10.3f} | {np.percentile(cir_e2e_latencies, 95):<10.3f} | {np.percentile(cir_e2e_latencies, 99):<10.3f}")
print("=" * 80)

p95_text = np.percentile(text_e2e_latencies, 95)
p95_cir = np.percentile(cir_e2e_latencies, 95)
print(f"🎯 KẾT LUẬN TIÊU CHUẨN KỸ THUẬT context.md (< 200 ms / query):")
print(f"   • Text-to-Image P95 Latency: {p95_text:.2f} ms  -> {'✅ ĐẠT YÊU CẦU' if p95_text < 200 else '❌ VƯỢT NGƯỠNG'}")
print(f"   • CIR Retrieval P95 Latency: {p95_cir:.2f} ms  -> {'✅ ĐẠT YÊU CẦU' if p95_cir < 200 else '❌ VƯỢT NGƯỠNG'}")
print(f"   • Tốc độ truy vấn FAISS:     {1000.0 / np.mean(faiss_latencies):.1f} queries / giây")
print("=" * 80)

---
## 🛍️ Phần 15: Thử Nghiệm Tìm Kiếm Thực Tế Top-10 Sản Phẩm (FAISS HNSW Retrieval)
Thực hiện truy vấn câu hỏi tiếng Việt tự nhiên và hiển thị danh sách Top-10 sản phẩm tương đồng nhất cùng độ tương đồng tương ứng.

In [ ]:
def search_catalog(query_text: str, top_k: int = 10):
    """Tìm kiếm Top-K sản phẩm bằng mô tả tiếng Việt."""
    model.eval()
    with torch.no_grad():
        inp = model.processor(text=[query_text], padding="max_length", truncation=True, max_length=getattr(model, "max_text_len", 16), return_tensors="pt")
        txt_emb = model.encode_text(inp["input_ids"].to(DEVICE), inp.get("attention_mask")).cpu().numpy()
        scores, indices = faiss_index.search(txt_emb, top_k)

    print("=" * 75)
    print(f"🔍 KẾT QUẢ TÌM KIẾM CHO TRUY VẤN: '{query_text}'")
    print("=" * 75)
    for rank, (score, idx) in enumerate(zip(scores[0], indices[0]), start=1):
        item = gallery_metadata[idx]
        print(f"[{rank:02d}] Tương đồng: {score:.4f} | Mã SP: {item.get('product_id')} | Danh mục: {item.get('category')}")
        print(f"     Tiêu đề: {item.get('title')}")
        if 'price' in item and item['price']:
            print(f"     Giá: {item['price']:,} VNĐ")
        print("-" * 75)

search_catalog("áo sơ mi trắng nam công sở dáng slim fit", top_k=10)

### 🌟 Bước Tiếp Theo: Xây Dựng Hệ Thống Chỉ Mục Tìm Kiếm Thực Tế (FAISS HNSW Index) Trên Bộ Ảnh Thật
Đoạn code dưới đây sử dụng mô hình SigLIP đã được Fine-tune để trích xuất vector đặc trưng thực tế từ tập dữ liệu sản phẩm, sau đó xây dựng cấu trúc chỉ mục đồ thị HNSW hiệu năng cao nhằm đảm bảo tốc độ truy vấn tối ưu nhất.

In [ ]:
import faiss
import numpy as np
import torch
from tqdm.auto import tqdm
import time
import json

print("-> Đang bắt đầu trích xuất Image Embeddings từ tập dữ liệu ảnh thật...")
model.eval()
gallery_embeddings = []
gallery_metadata = []
processor = model.processor

with torch.no_grad():
    for batch in tqdm(test_loader, desc="Trích xuất đặc trưng ảnh"):
        # Tiền xử lý ảnh qua processor
        inputs = processor(images=batch["images"], return_tensors="pt")
        pixel_values = inputs["pixel_values"].to(DEVICE)

        # Trích xuất vector đặc trưng
        with torch.amp.autocast(device_type="cuda", dtype=AMP_DTYPE):
            img_feats = model.encode_image(pixel_values)

        gallery_embeddings.append(img_feats.cpu().numpy())

        # Lưu trữ metadata thực tế của ảnh
        for i in range(len(batch["product_ids"])):
            gallery_metadata.append({
                "product_id": batch["product_ids"][i],
                "title": batch["texts"][i],
                "category": batch["categories"][i]
            })

# Chuyển đổi sang numpy array và chuẩn hóa L2
gallery_embeddings = np.concatenate(gallery_embeddings, axis=0).astype(np.float32)
faiss.normalize_L2(gallery_embeddings)

# Thiết lập chỉ mục FAISS HNSW
dim = gallery_embeddings.shape[1]
M = 32
efConstruction = 128
efSearch = 64

real_faiss_index = faiss.IndexHNSWFlat(dim, M, faiss.METRIC_INNER_PRODUCT)
real_faiss_index.hnsw.efConstruction = efConstruction
real_faiss_index.hnsw.efSearch = efSearch

t_start = time.perf_counter()
real_faiss_index.add(gallery_embeddings)
t_end = time.perf_counter()

print(f"\n✅ Đã trích xuất thành công {len(gallery_embeddings):,} sản phẩm thực tế!")
print(f"✅ Đã xây dựng hoàn tất FAISS HNSW Index trong {(t_end - t_start):.3f} giây!")

### 🔍 Thử Nghiệm Truy Vấn Trực Tiếp Trên Cơ Sở Dữ Liệu Ảnh Thật
Bây giờ chúng ta sẽ định nghĩa hàm tìm kiếm thực tế và chạy kiểm thử với câu truy vấn của bạn để theo dõi độ chính xác ngữ nghĩa và tốc độ phản hồi.

In [ ]:
def search_real_catalog(query_text: str, top_k: int = 5):
    """Tìm kiếm sản phẩm thật bằng văn bản tiếng Việt sử dụng FAISS HNSW."""
    model.eval()
    with torch.no_grad():
        # Tokenize và trích xuất vector truy vấn từ văn bản
        inp = model.processor(text=[query_text], padding="max_length", truncation=True, max_length=128, return_tensors="pt")
        txt_emb = model.encode_text(inp["input_ids"].to(DEVICE), inp.get("attention_mask")).cpu().numpy()

        # Tìm kiếm các láng giềng gần nhất trên đồ thị HNSW
        t_start = time.perf_counter()
        scores, indices = real_faiss_index.search(txt_emb, top_k)
        latency = (time.perf_counter() - t_start) * 1000.0

    print("=" * 85)
    print(f"🔍 KẾT QUẢ TÌM KIẾM THỰC TẾ CHO TRUY VẤN: '{query_text}'")
    print(f"⏱️ Thời gian truy vấn Index: {latency:.2f} ms")
    print("=" * 85)
    for rank, (score, idx) in enumerate(zip(scores[0], indices[0]), start=1):
        item = gallery_metadata[idx]
        print(f"[{rank:02d}] Độ tương đồng: {score:.4f} | Nhóm: {item.get('category')}")
        print(f"     Tiêu đề: {item.get('title')}")
        print("-" * 85)

# Chạy thử nghiệm truy vấn thực tế
search_real_catalog("áo sơ mi nam công sở tay dài màu trắng cổ đức", top_k=5)

In [ ]:
def run_hyperparameter_tuning_experiment(
    model_instance,
    train_loader_instance,
    val_loader_instance,
    learning_rate=1e-4,
    backbone_lr_ratio=0.1,
    weight_decay=2.5e-2,
    epochs=3
):
    """
    Hàm chạy thử nghiệm nhanh (3 epochs) cấu hình siêu tham số mới đề xuất để kiểm tra
    khả năng hội tụ và tốc độ tăng trưởng của mAP.
    """
    print("=== 🚀 BẮT ĐẦU THỬ NGHIỆM TỐI ƯU HÓA SIÊU THAM SỐ ===")
    print(f"   • LR chính (Combiner/Heads): {learning_rate:.2e}")
    print(f"   • LR Backbone (SigLIP):     {learning_rate * backbone_lr_ratio:.2e}")
    print(f"   • Weight Decay:            {weight_decay:.2e}")
    print(f"   • Số Epochs chạy thử:      {epochs}")

    # 1. Khởi tạo hàm loss Ultra nâng cao
    tuned_loss_fn = UltraECommerceLoss(margin=0.28, weight_ms=1.2)

    # 2. Phân tách tỷ lệ học tập (Differential Learning Rates)
    backbone_params = [p for name, p in model_instance.named_parameters() if "siglip" in name and p.requires_grad]
    combiner_params = [p for name, p in model_instance.named_parameters() if "cir_combiner" in name or "loss_fn" in name]

    tuned_optimizer = torch.optim.AdamW([
        {"params": backbone_params, "lr": learning_rate * backbone_lr_ratio},
        {"params": combiner_params, "lr": learning_rate}
    ], weight_decay=weight_decay)

    # 3. Tạo Trainer cục bộ cho thử nghiệm
    experiment_trainer = MultimodalTrainer(
        model=model_instance,
        train_loader=train_loader_instance,
        val_loader=val_loader_instance,
        test_loader=None,
        loss_fn=tuned_loss_fn,
        device=DEVICE,
        batch_size=BATCH_SIZE,
        learning_rate=learning_rate,
        weight_decay=weight_decay,
        max_epochs=epochs,
        warmup_steps=100,
        checkpoint_dir=f"{CHECKPOINT_DIR}/exp_tuned",
        patience=3
    )

    # Thay thế optimizer mặc định bằng optimizer biệt hóa đã tuned
    experiment_trainer.optimizer = tuned_optimizer

    # 4. Thực thi huấn luyện
    for ep in range(1, epochs + 1):
        start_t = time.time()
        train_loss = experiment_trainer.train_epoch(ep)
        val_metrics = experiment_trainer.evaluate(experiment_trainer.val_loader, desc=f"Exp Epoch {ep} [Val]")

        duration = time.time() - start_t
        print(f"⭐ Epoch [{ep}/{epochs}] | Thời gian: {duration:.1f}s | Train Loss: {train_loss:.4f} | Val Loss: {val_metrics['loss']:.4f} | Val mAP: {val_metrics['mAP']:.4f} (Recall@10: {val_metrics['Recall@10']:.2f}%)")

    print("\n✅ Thử nghiệm hoàn tất! Kiểm tra xem Val mAP có cải thiện nhanh hơn cấu hình cũ không.")
    return experiment_trainer

# Chạy thử nghiệm ngay tại chỗ với 1 epoch thử nghiệm
# run_hyperparameter_tuning_experiment(model, train_loader, val_loader, epochs=1)

In [ ]:
# Chạy thử nghiệm nhanh cấu hình tối ưu hóa siêu tham số mới trong 1 epoch
tuned_trainer = run_hyperparameter_tuning_experiment(
    model_instance=model,
    train_loader_instance=train_loader,
    val_loader_instance=val_loader,
    learning_rate=1e-4,
    backbone_lr_ratio=0.1,
    weight_decay=2.5e-2,
    epochs=1
)

# 📈 Chiến Lược Đột Phá Chỉ Số mAP & Recall cho Hệ Thống Khuyến Nghị

Để khắc phục hiện tượng trả kết quả sai lệch và tăng điểm số **mAP** từ mức hiện tại (~0.31) lên **0.65+**, bạn nên cấu hình huấn luyện tiếp tục với các siêu tham số tối ưu sau:

| Siêu tham số | Giá trị đề xuất | Vai trò tác động đến mAP |
| :--- | :--- | :--- |
| **Learning Rate (Backbone)** | `5e-6` đến `1e-5` | Giữ nguyên khả năng trích xuất đặc trưng thị giác cao cấp của SigLIP ViT-SO400M. |
| **Learning Rate (Combiner/Heads)** | `1e-4` | Đẩy nhanh tốc độ căn chỉnh ngữ nghĩa đa phương thức tiếng Việt. |
| **Loss Function** | `UltraECommerceLoss` | Phạt mềm và đào bới mẫu tiêu cực khó (Hard Negatives), loại bỏ nhiễu danh mục. |
| **Epochs tối thiểu** | `10 - 15 epochs` | Cho phép các Projection Head hội tụ hoàn toàn vào không gian siêu cầu đơn vị. |
| **Text Template** | `[Danh mục] Tiêu đề + Thuộc tính (Màu, dáng) + Caption` | Cung cấp ngữ cảnh phong phú tránh mô hình tìm kiếm mù mờ. |

---
## 💾 Phần 16: Tự Động Sao Lưu Toàn Bộ Notebook & Kết Quả Lên Google Drive
Đồng bộ tệp notebook, trọng số mô hình, siêu dữ liệu và chỉ mục tìm kiếm vào Google Drive (`/content/drive/MyDrive/Hoc_sau_ver_3/`) để bảo toàn kết quả lâu dài.

In [ ]:
# 1. Đường dẫn thư mục sao lưu notebook trên Drive
DRIVE_NOTEBOOK_DIR = f"{DRIVE_DIR}/notebooks"
os.makedirs(DRIVE_NOTEBOOK_DIR, exist_ok=True)

# 2. Tự động sao chép các file notebook đang chạy vào Drive
colab_nb_files = glob.glob("/content/*.ipynb")
for nb_file in colab_nb_files:
    dest_path = os.path.join(DRIVE_NOTEBOOK_DIR, os.path.basename(nb_file))
    shutil.copy2(nb_file, dest_path)
    print(f"☁️ Đã sao lưu Notebook sang Google Drive: {dest_path}")

print("=" * 80)
print("🎉 TOÀN BỘ DỰ ÁN ĐÃ ĐƯỢC SAO LƯU AN TOÀN TRÊN GOOGLE DRIVE:")
print(f"   • Trọng số mô hình tối ưu:     {DRIVE_CHECKPOINT_DIR}/best_model.pt")
print(f"   • Lịch sử huấn luyện JSON:     {DRIVE_CHECKPOINT_DIR}/train_history.json")
print(f"   • Chỉ mục tìm kiếm FAISS HNSW: {DRIVE_CHECKPOINT_DIR}/faiss_hnsw_index.bin")
print(f"   • Siêu dữ liệu sản phẩm:       {DRIVE_CHECKPOINT_DIR}/product_metadata.json")
print(f"   • Thư mục Notebook sao lưu:    {DRIVE_NOTEBOOK_DIR}/")
print("=" * 80)